# Calculadora Principal

## Ejecución automática en Google Colab

Ejecuta la celda siguiente con el botón ▶. El cuaderno genera la aplicación, instala las dependencias, inicia Streamlit y despliega la calculadora dentro de esta misma página. No necesitas abrir una terminal ni ejecutar `streamlit run app.py` manualmente.

**Importante:** mantén activa la sesión de Colab mientras utilizas la aplicación.


In [ ]:
# ============================================================
# LANZADOR ROBUSTO PARA GOOGLE COLAB + STREAMLIT
# ============================================================
from pathlib import Path
import sys, subprocess, time, socket, os, urllib.request
from google.colab import output

APP_SOURCE = 'import streamlit as st\nimport pandas as pd\nimport numpy as np\nimport plotly.express as px\nimport plotly.graph_objects as go\nfrom plotly.subplots import make_subplots\nfrom scipy import stats\nimport statsmodels.api as sm\nfrom statsmodels.formula.api import ols\nfrom statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison\nimport warnings\nimport requests\nimport io\nimport re\nfrom itertools import combinations, product\nwarnings.filterwarnings(\'ignore\')\n\nst.set_page_config(\n    page_title="📊 Diseños Experimentales y ANOVA",\n    page_icon="📊",\n    layout="wide"\n)\n\nst.title("📊 Calculadora de Diseños Experimentales y ANOVA")\nst.markdown("---")\n\n# ============================================================\n# INICIALIZAR SESSION STATE\n# ============================================================\n\nif \'datos_cargados\' not in st.session_state:\n    st.session_state.datos_cargados = {}\nif \'resultados_anova\' not in st.session_state:\n    st.session_state.resultados_anova = {}\nif \'diseno_actual\' not in st.session_state:\n    st.session_state.diseno_actual = None\nif \'factores_definidos\' not in st.session_state:\n    st.session_state.factores_definidos = {}\nif \'respuestas_definidas\' not in st.session_state:\n    st.session_state.respuestas_definidas = {}\nif \'pruebas_multiples\' not in st.session_state:\n    st.session_state.pruebas_multiples = {}\n\n# Variables exclusivas para la pestaña Fraccionado\nif \'fr_df\' not in st.session_state: st.session_state.fr_df = None\nif \'fr_analizado\' not in st.session_state: st.session_state.fr_analizado = False\nif \'fr_colapsado\' not in st.session_state: st.session_state.fr_colapsado = None\n\n# ============================================================\n# FUNCIONES AUXILIARES\n# ============================================================\n\ndef selector_tipo_anova(key_suffix="", show_help=True):\n    if show_help:\n        st.markdown("""\n        **📊 Tipos de ANOVA:**\n        - **Tipo I (Secuencial):** El orden de entrada importa.\n        - **Tipo II (Parcial):** Recomendado para diseños balanceados.\n        - **Tipo III (Parcial con interacción):** Útil para diseños no balanceados.\n        """)\n    tipo_anova = st.radio(\n        "Selecciona el tipo de ANOVA:",\n        ["Tipo I (Secuencial)", "Tipo II (Parcial)", "Tipo III (Parcial con interacción)"],\n        horizontal=True, key=f"tipo_anova_{key_suffix}"\n    )\n    tipo_map = {"Tipo I (Secuencial)": 1, "Tipo II (Parcial)": 2, "Tipo III (Parcial con interacción)": 3}\n    return tipo_map[tipo_anova]\n\ndef analizar_anova_mejorada(df, formula, titulo="ANOVA", key_suffix="", tipo="dca", anova_type=2):\n    try:\n        for col in df.select_dtypes(include=[\'object\']).columns:\n            df[col] = df[col].astype(\'category\')\n        modelo = ols(formula, data=df).fit()\n        anova_table = sm.stats.anova_lm(modelo, typ=anova_type)\n        if \'mean_sq\' not in anova_table.columns:\n            anova_table[\'mean_sq\'] = anova_table[\'sum_sq\'] / anova_table[\'df\']\n        rename_map = {\'sum_sq\': \'Sum Sq\', \'df\': \'DF\', \'mean_sq\': \'Mean Sq\', \'F\': \'F-Ratio\', \'PR(>F)\': \'P-Value\'}\n        anova_table = anova_table.rename(columns=rename_map)\n        columnas_orden = [c for c in anova_table.columns if c not in [\'Mean Sq\']]\n        if \'DF\' in columnas_orden:\n            idx_df = columnas_orden.index(\'DF\')\n            columnas_orden.insert(idx_df + 1, \'Mean Sq\')\n            anova_table = anova_table[columnas_orden]\n\n        # Guardamos el resultado\n        st.session_state.resultados_anova[key_suffix] = {\n            \'modelo\': modelo, \'anova\': anova_table, \'df\': df, \'tipo\': tipo,\n            \'formula\': formula, \'anova_type\': anova_type, \'titulo\': titulo\n        }\n\n        st.success(f"✅ {titulo} calculado (Tipo {anova_type})")\n        st.dataframe(anova_table.style.format({\n            \'Sum Sq\': \'{:.4f}\', \'Mean Sq\': \'{:.4f}\', \'F-Ratio\': \'{:.4f}\', \'P-Value\': \'{:.6f}\'\n        }, na_rep="None"))\n        st.write(f"**R² = {modelo.rsquared:.6f}**")\n        st.write(f"**R² ajustado = {modelo.rsquared_adj:.6f}**")\n        return modelo, anova_table\n    except Exception as e:\n        st.error(f"❌ Error en ANOVA: {str(e)}")\n        return None, None\n\ndef mostrar_tabla_anova(key_suffix):\n    """Imprime de forma persistente la tabla ANOVA guardada en session_state."""\n    if key_suffix in st.session_state.resultados_anova:\n        res = st.session_state.resultados_anova[key_suffix]\n        st.success(f"✅ {res.get(\'titulo\', \'ANOVA\')} calculado (Tipo {res.get(\'anova_type\', 2)})")\n        st.dataframe(res[\'anova\'].style.format({\n            \'Sum Sq\': \'{:.4f}\', \'Mean Sq\': \'{:.4f}\', \'F-Ratio\': \'{:.4f}\', \'P-Value\': \'{:.6f}\'\n        }, na_rep="None"))\n        st.write(f"**R² = {res[\'modelo\'].rsquared:.6f}** | **R² ajustado = {res[\'modelo\'].rsquared_adj:.6f}**")\n\ndef graficos_interaccion_multifactorial(df, factores, respuesta, max_pares=6):\n    if len(factores) < 2:\n        return None\n    pares = list(combinations(factores, 2))[:max_pares]\n    n_filas = (len(pares) + 1) // 2\n    fig = make_subplots(rows=n_filas, cols=2, subplot_titles=[f"{f1} × {f2}" for f1, f2 in pares])\n    for idx, (f1, f2) in enumerate(pares):\n        row = idx // 2 + 1\n        col = idx % 2 + 1\n        medias = df.groupby([f1, f2])[respuesta].mean().reset_index()\n        for nivel in medias[f2].unique():\n            datos_nivel = medias[medias[f2] == nivel]\n            fig.add_trace(go.Scatter(x=datos_nivel[f1], y=datos_nivel[respuesta],\n                                     mode=\'lines+markers\', name=f"{f2}={nivel}",\n                                     legendgroup=f"{f2}={nivel}", showlegend=(idx == 0)),\n                          row=row, col=col)\n        fig.update_xaxes(title_text=f1, row=row, col=col)\n        fig.update_yaxes(title_text=respuesta, row=row, col=col)\n    fig.update_layout(height=min(400, 200 * n_filas), showlegend=True,\n                      title_text="Gráficos de Interacción")\n    return fig\n\ndef graficos_interaccion_heatmap(df, factores, respuesta):\n    if len(factores) < 2:\n        return None\n    try:\n        f1, f2 = factores[0], factores[1]\n        pivot = df.pivot_table(index=f1, columns=f2, values=respuesta, aggfunc=\'mean\')\n        fig = px.imshow(pivot, text_auto=True, aspect="auto",\n                        color_continuous_scale=\'RdBu_r\', title=f"Mapa de Calor: {f1} × {f2}")\n        fig.update_layout(height=400)\n        return fig\n    except:\n        return None\n\ndef graficar_boxplot_por_factor(df, factores, respuesta):\n    if not factores:\n        return None\n    n_factores = len(factores)\n    fig = make_subplots(rows=1, cols=n_factores, subplot_titles=[f"{f}" for f in factores])\n    for i, factor in enumerate(factores):\n        fig.add_trace(\n            go.Box(y=df[respuesta], x=df[factor], name=factor,\n                   marker_color=\'#3498db\', boxpoints=\'all\', jitter=0.3, pointpos=-1.8),\n            row=1, col=i+1\n        )\n        fig.update_xaxes(title_text=factor, row=1, col=i+1)\n        fig.update_yaxes(title_text=respuesta, row=1, col=i+1)\n    fig.update_layout(height=400, showlegend=False, title_text="Boxplot por Factor")\n    return fig\n\ndef graficar_sumas_cuadrados(anova_table):\n    try:\n        df_sc = anova_table.reset_index().rename(columns={\'index\': \'Fuente\'})\n        col_sc = \'Sum Sq\' if \'Sum Sq\' in df_sc.columns else \'sum_sq\' if \'sum_sq\' in df_sc.columns else None\n        if not col_sc:\n            return None\n        if \'Total\' in df_sc[\'Fuente\'].values:\n            df_sc = df_sc[df_sc[\'Fuente\'] != \'Total\']\n        fig = px.bar(df_sc, x=\'Fuente\', y=col_sc, title="Gráfico de Sumas de Cuadrados",\n                     color=\'Fuente\', text_auto=\'.2f\')\n        if df_sc[col_sc].max() / max(df_sc[col_sc].min(), 0.001) > 100:\n            fig.update_yaxes(type="log")\n        fig.update_layout(showlegend=False, height=400)\n        fig.update_traces(textposition=\'outside\')\n        return fig\n    except:\n        return None\n\ndef graficar_verificacion_modelo_completa(modelo, df, factor, respuesta):\n    if modelo is None:\n        return None\n    residuos = modelo.resid\n    predichos = modelo.fittedvalues\n    run_order = np.arange(1, len(residuos) + 1)\n    fig = make_subplots(rows=2, cols=2, subplot_titles=[\n        "Q-Q (Normalidad)", "Residuos vs Ajustados",\n        "Residuos vs Orden", "Histograma de Residuos"\n    ])\n    from scipy.stats import probplot\n    (osm, osr), (slope, intercept, r) = probplot(residuos, dist="norm", fit=True)\n    fig.add_trace(go.Scatter(x=osm, y=osr, mode=\'markers\', marker=dict(color=\'blue\', size=6)), row=1, col=1)\n    fig.add_trace(go.Scatter(x=[min(osm), max(osm)],\n                             y=[min(osm)*slope + intercept, max(osm)*slope + intercept],\n                             mode=\'lines\', line=dict(color=\'red\', dash=\'dash\')), row=1, col=1)\n    fig.add_trace(go.Scatter(x=predichos, y=residuos, mode=\'markers\', marker=dict(color=\'green\', size=6)), row=1, col=2)\n    fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)\n    fig.add_trace(go.Scatter(x=run_order, y=residuos, mode=\'lines+markers\', marker=dict(color=\'orange\', size=6)), row=2, col=1)\n    fig.add_hline(y=0, line_dash="dash", line_color="red", row=2, col=1)\n    fig.add_trace(go.Histogram(x=residuos, marker_color=\'purple\', nbinsx=15), row=2, col=2)\n    fig.update_layout(height=700, showlegend=False, title_text="Verificación de Supuestos")\n    return fig\n\ndef graficar_interaccion_mejorada(df, f1, f2, respuesta):\n    medias = df.groupby([f1, f2])[respuesta].mean().reset_index()\n    fig = px.line(medias, x=f1, y=respuesta, color=f2, markers=True, title=f"Interacción {f1} × {f2}")\n    for i, row in medias.iterrows():\n        fig.add_annotation(x=row[f1], y=row[respuesta], text=f"{row[respuesta]:.2f}",\n                           showarrow=False, yshift=10, font=dict(size=10))\n    fig.update_layout(height=400)\n    return fig\n\ndef graficar_efectos_e_interacciones(df, factores, resp):\n    """Genera gráficos de líneas para Efectos Principales e Interacciones de 2 vías."""\n    st.markdown("---")\n    st.subheader("📈 Gráficos de Efectos Principales")\n    cols = st.columns(min(len(factores), 3))\n    for i, f in enumerate(factores):\n        with cols[i % len(cols)]:\n            medias = df.groupby(f)[resp].mean().reset_index()\n            fig = px.line(medias, x=f, y=resp, markers=True, title=f"Efecto Principal: {f}")\n            fig.update_layout(height=250, showlegend=False)\n            st.plotly_chart(fig, use_container_width=True)\n\n    if len(factores) >= 2:\n        st.subheader("🔄 Gráficos de Interacciones (Pares)")\n        pares = list(combinations(factores, 2))\n        if len(pares) > 6:\n            st.caption("Mostrando las primeras 6 interacciones...")\n            pares = pares[:6]\n\n        cols_int = st.columns(2)\n        for i, (f1, f2) in enumerate(pares):\n            with cols_int[i % 2]:\n                medias = df.groupby([f1, f2])[resp].mean().reset_index()\n                fig = px.line(medias, x=f1, y=resp, color=f2, markers=True, title=f"Interacción: {f1} × {f2}")\n                fig.update_layout(height=300)\n                st.plotly_chart(fig, use_container_width=True)\n\n# ============================================================\n# ⭐ NUEVO: SUMMARY STATISTICS (ESTILO STATGRAPHICS)\n# ============================================================\n\ndef calcular_summary_statistics(modelo, df, factores, respuesta, alpha=0.05):\n    from scipy.stats import t as t_dist\n    try:\n        anova_table = sm.stats.anova_lm(modelo, typ=2)\n        if \'Residual\' in anova_table.index:\n            mse = anova_table.loc[\'Residual\', \'sum_sq\'] / anova_table.loc[\'Residual\', \'df\']\n            gl_error = anova_table.loc[\'Residual\', \'df\']\n        else:\n            mse = modelo.mse_resid\n            gl_error = modelo.df_resid\n    except Exception:\n        mse = modelo.mse_resid\n        gl_error = modelo.df_resid\n\n    t_crit = t_dist.ppf(1 - alpha / 2, gl_error) if gl_error > 0 else np.nan\n\n    filas = []\n    grand_mean = df[respuesta].mean()\n    n_total = len(df)\n    se_grand = np.sqrt(mse / n_total) if pd.notna(mse) and mse > 0 else np.nan\n    filas.append({\n        \'Level\': \'GRAND MEAN\', \'Count\': n_total, \'Mean\': grand_mean,\n        \'Stnd. Error\': se_grand,\n        \'Lower Limit\': grand_mean - t_crit * se_grand if pd.notna(t_crit) else np.nan,\n        \'Upper Limit\': grand_mean + t_crit * se_grand if pd.notna(t_crit) else np.nan,\n    })\n\n    for factor in factores:\n        filas.append({\'Level\': factor, \'Count\': None, \'Mean\': None,\n                      \'Stnd. Error\': None, \'Lower Limit\': None, \'Upper Limit\': None})\n        try:\n            niveles = sorted(df[factor].dropna().unique(), key=lambda x: (str(type(x)), str(x)))\n        except Exception:\n            niveles = list(df[factor].dropna().unique())\n        if len(niveles) > 30:\n            filas.append({\'Level\': \'   (factor continuo)\', \'Count\': None, \'Mean\': None,\n                          \'Stnd. Error\': None, \'Lower Limit\': None, \'Upper Limit\': None})\n            continue\n        for nivel in niveles:\n            sub = df[df[factor] == nivel]\n            n = len(sub)\n            if n == 0:\n                continue\n            media = sub[respuesta].mean()\n            se = np.sqrt(mse / n) if pd.notna(mse) and mse > 0 else np.nan\n            filas.append({\n                \'Level\': str(nivel), \'Count\': n, \'Mean\': media, \'Stnd. Error\': se,\n                \'Lower Limit\': media - t_crit * se if pd.notna(t_crit) else np.nan,\n                \'Upper Limit\': media + t_crit * se if pd.notna(t_crit) else np.nan,\n            })\n\n    return pd.DataFrame(filas), mse, gl_error, t_crit\n\ndef mostrar_summary_statistics(modelo, df, factores, respuesta, key_suffix="", alpha=0.05, incluir_descriptivas=True):\n    st.subheader("📋 Summary Statistics — Least Squares Means (95% CI)")\n    try:\n        tabla, mse, gl_error, t_crit = calcular_summary_statistics(modelo, df, factores, respuesta, alpha)\n        tabla_fmt = tabla.copy()\n        for col in [\'Mean\', \'Stnd. Error\', \'Lower Limit\', \'Upper Limit\']:\n            tabla_fmt[col] = tabla_fmt[col].apply(lambda x: f"{x:.6f}" if pd.notna(x) and isinstance(x, (int, float, np.floating)) else "")\n        tabla_fmt[\'Count\'] = tabla_fmt[\'Count\'].apply(lambda x: str(int(x)) if pd.notna(x) and x is not None and x != "" else "")\n\n        st.caption(\n            f"**MSE = {mse:.6f}  |  GL Error = {int(gl_error)}  |  "\n            f"Nivel de confianza = {(1-alpha)*100:.1f}%  |  t crítico = {t_crit:.4f}**"\n        )\n        st.dataframe(tabla_fmt, use_container_width=True, hide_index=True)\n\n        csv = tabla.to_csv(index=False).encode(\'utf-8\')\n        st.download_button("📥 Descargar tabla LS Means (CSV)", data=csv, file_name=f"lsmeans_{key_suffix}.csv", mime="text/csv", key=f"dl_ss_{key_suffix}")\n\n        if incluir_descriptivas:\n            with st.expander("📈 Estadísticas descriptivas (media, mediana, moda, cuartiles, CV)", expanded=False):\n                filas_desc = []\n                for factor in factores:\n                    try:\n                        niveles = sorted(df[factor].dropna().unique(), key=lambda x: (str(type(x)), str(x)))\n                    except Exception:\n                        niveles = list(df[factor].dropna().unique())\n                    if len(niveles) > 30: continue\n                    for nivel in niveles:\n                        s = df[df[factor] == nivel][respuesta]\n                        moda_vals = s.mode()\n                        filas_desc.append({\n                            \'Factor\': factor, \'Nivel\': str(nivel), \'N\': len(s),\n                            \'Media\': s.mean(), \'Mediana\': s.median(),\n                            \'Moda\': moda_vals.iloc[0] if len(moda_vals) > 0 else np.nan,\n                            \'Desv.Est.\': s.std(), \'Varianza\': s.var(),\n                            \'Mínimo\': s.min(), \'Q1\': s.quantile(0.25),\n                            \'Q3\': s.quantile(0.75), \'Máximo\': s.max(),\n                            \'CV (%)\': (s.std() / s.mean()) * 100 if s.mean() != 0 else np.nan,\n                        })\n                s = df[respuesta]\n                moda_vals = s.mode()\n                filas_desc.insert(0, {\n                    \'Factor\': \'GLOBAL\', \'Nivel\': \'—\', \'N\': len(s),\n                    \'Media\': s.mean(), \'Mediana\': s.median(),\n                    \'Moda\': moda_vals.iloc[0] if len(moda_vals) > 0 else np.nan,\n                    \'Desv.Est.\': s.std(), \'Varianza\': s.var(),\n                    \'Mínimo\': s.min(), \'Q1\': s.quantile(0.25),\n                    \'Q3\': s.quantile(0.75), \'Máximo\': s.max(),\n                    \'CV (%)\': (s.std() / s.mean()) * 100 if s.mean() != 0 else np.nan,\n                })\n                df_desc = pd.DataFrame(filas_desc)\n                st.dataframe(\n                    df_desc.style.format({\n                        \'Media\': \'{:.4f}\', \'Mediana\': \'{:.4f}\', \'Moda\': \'{:.4f}\',\n                        \'Desv.Est.\': \'{:.4f}\', \'Varianza\': \'{:.4f}\',\n                        \'Mínimo\': \'{:.4f}\', \'Q1\': \'{:.4f}\', \'Q3\': \'{:.4f}\',\n                        \'Máximo\': \'{:.4f}\', \'CV (%)\': \'{:.2f}\'\n                    }),\n                    use_container_width=True, hide_index=True\n                )\n                st.download_button("📥 Descargar descriptivas (CSV)", data=df_desc.to_csv(index=False).encode(\'utf-8\'), file_name=f"descriptivas_{key_suffix}.csv", mime="text/csv", key=f"dl_desc_{key_suffix}")\n    except Exception as e:\n        st.warning(f"⚠️ No se pudieron calcular las estadísticas: {e}")\n\n# ============================================================\n# PRUEBAS MÚLTIPLES Y RESIDUOS\n# ============================================================\n\ndef realizar_tukey(df, factor, respuesta, alpha=0.05):\n    try:\n        df[factor] = df[factor].astype(\'category\')\n        return pairwise_tukeyhsd(df[respuesta], df[factor], alpha=alpha)\n    except: return None\n\ndef interpretar_tukey(tukey):\n    interpretacion = []\n    if tukey is None: return ["❌ No se pudo realizar la prueba de Tukey"]\n    try:\n        tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])\n        significativas = tukey_df[tukey_df[\'reject\'] == True]\n        if len(significativas) == 0:\n            interpretacion.append("📊 **Tukey HSD:** No hay diferencias significativas")\n            return interpretacion\n        interpretacion.append(f"📊 **Tukey HSD:** {len(significativas)} diferencias significativas")\n        for _, row in significativas.iterrows():\n            interpretacion.append(f"  • {row[\'group1\']} vs {row[\'group2\']}: Diff = {row[\'meandiff\']:.4f} (p = {row[\'p-adj\']:.6f})")\n    except Exception as e:\n        interpretacion.append(f"❌ Error: {str(e)}")\n    return interpretacion\n\ndef realizar_lsd(df, factor, respuesta, mse, gl_error, alpha=0.05):\n    try:\n        if mse is None or gl_error is None or gl_error == 0: return None\n        medias = df.groupby(factor)[respuesta].mean()\n        grupos = medias.index.tolist()\n        t_crit = stats.t.ppf(1 - alpha/2, gl_error)\n        resultados = []\n        for i in range(len(grupos)):\n            for j in range(i+1, len(grupos)):\n                g1, g2 = grupos[i], grupos[j]\n                diff = medias[g1] - medias[g2]\n                n1 = len(df[df[factor] == g1])\n                n2 = len(df[df[factor] == g2])\n                se_diff = np.sqrt(mse * (1/n1 + 1/n2))\n                t_val = diff / se_diff\n                p_val = 2 * (1 - stats.t.cdf(abs(t_val), gl_error))\n                lsd = t_crit * se_diff\n                significativo = p_val < alpha\n                resultados.append({\n                    \'group1\': g1, \'group2\': g2, \'meandiff\': diff,\n                    \'p-adj\': p_val, \'lower\': diff - lsd, \'upper\': diff + lsd,\n                    \'reject\': significativo\n                })\n        return pd.DataFrame(resultados)\n    except Exception as e: return None\n\ndef interpretar_lsd(lsd_df):\n    interpretacion = []\n    if lsd_df is None or len(lsd_df) == 0: return ["❌ No se pudo realizar la prueba de LSD"]\n    try:\n        significativas = lsd_df[lsd_df[\'reject\'] == True]\n        if len(significativas) == 0:\n            interpretacion.append("📊 **LSD:** No hay diferencias significativas")\n            return interpretacion\n        interpretacion.append(f"📊 **LSD:** {len(significativas)} diferencias significativas")\n        for _, row in significativas.iterrows():\n            interpretacion.append(f"  • {row[\'group1\']} vs {row[\'group2\']}: Diff = {row[\'meandiff\']:.4f} (p = {row[\'p-adj\']:.6f})")\n    except Exception as e: interpretacion.append(f"❌ Error: {str(e)}")\n    return interpretacion\n\ndef extraer_mse_gl(anova_table):\n    try:\n        if \'Residual\' in anova_table.index: return anova_table.loc[\'Residual\', \'Mean Sq\'], anova_table.loc[\'Residual\', \'DF\']\n        else: return anova_table[\'Mean Sq\'].iloc[-1], anova_table[\'DF\'].iloc[-1]\n    except: return None, None\n\ndef mostrar_graficos_residuos_directo(modelo, df, factor=None, respuesta=None, key_suffix="", titulo="Residuos"):\n    if modelo is None: return\n    st.subheader(f"📈 {titulo}")\n    with st.spinner("Generando gráficos de verificación del modelo..."):\n        fig = graficar_verificacion_modelo_completa(modelo, df, factor, respuesta)\n        if fig: st.plotly_chart(fig, use_container_width=True)\n        st.subheader("📊 Pruebas de Normalidad de Residuos")\n        from scipy.stats import shapiro, anderson\n        residuos = modelo.resid\n        col1, col2 = st.columns(2)\n        with col1:\n            try:\n                shapiro_stat, shapiro_p = shapiro(residuos)\n                st.metric("Shapiro-Wilk", f"p = {shapiro_p:.6f}")\n                if shapiro_p > 0.05: st.success("✅ Los residuos siguen distribución normal (p > 0.05)")\n                else: st.warning("⚠️ Los residuos NO siguen distribución normal (p ≤ 0.05)")\n            except: st.warning("⚠️ No se pudo realizar la prueba de Shapiro-Wilk")\n        with col2:\n            try:\n                anderson_result = anderson(residuos)\n                st.metric("Anderson-Darling", f"Stat = {anderson_result.statistic:.4f}")\n                if anderson_result.statistic < anderson_result.critical_values[2]: st.success("✅ Los residuos siguen distribución normal (al 5%)")\n                else: st.warning("⚠️ Los residuos NO siguen distribución normal (al 5%)")\n            except: st.warning("⚠️ No se pudo realizar la prueba de Anderson-Darling")\n\n# ============================================================\n# ⭐ NUEVO: FUNCIONES GLOBALES PARA PARETO, DANIEL Y EXCLUIR\n# ============================================================\n\ndef mostrar_pareto_y_daniel(modelo, anova, resp, n_corridas):\n    st.markdown("---")\n    st.subheader("📊 Gráficos para Selección de Efectos (Pareto y Daniel)")\n    try:\n        def clean_name(name):\n            return name.replace(\'C(\', \'\').replace(\')\', \'\').replace(\'Factor_\', \'\').replace(\':\', \'\')\n        coefs = modelo.params.drop(\'Intercept\', errors=\'ignore\')\n        names = [clean_name(x) for x in coefs.index]\n        effects = 2 * coefs.values\n        gl_error = anova.loc[\'Residual\', \'DF\'] if \'Residual\' in anova.index else 0\n\n        if gl_error > 0:\n            mse = anova.loc[\'Residual\', \'Mean Sq\']\n            se = np.sqrt(mse * (4 / n_corridas))\n            limite = stats.t.ppf(0.975, gl_error) * se\n        else:\n            s0 = 1.5 * np.median(np.abs(effects))\n            reduced = effects[np.abs(effects) < 2.5 * s0]\n            pse = 1.5 * np.median(np.abs(reduced)) if len(reduced) > 0 else 1.5 * np.median(np.abs(effects))\n            limite = 2.0 * pse\n\n        col1, col2 = st.columns(2)\n        with col1:\n            st.markdown("**Diagrama de Pareto**")\n            df_pareto = pd.DataFrame({\'Efecto\': names, \'Magnitud\': np.abs(effects)}).sort_values(by=\'Magnitud\', ascending=True)\n            colors = [\'#2b8cbe\' if m > limite else \'#a6bddb\' for m in df_pareto[\'Magnitud\']]\n            fig_pareto = go.Figure(go.Bar(x=df_pareto[\'Magnitud\'], y=df_pareto[\'Efecto\'], orientation=\'h\', marker_color=colors))\n            fig_pareto.add_vline(x=limite, line_dash="dash", line_color="red")\n            fig_pareto.update_layout(xaxis_title="Efecto (|E|)", height=400, showlegend=False)\n            st.plotly_chart(fig_pareto, use_container_width=True)\n        with col2:\n            st.markdown("**Gráfico de Daniel**")\n            idx = np.argsort(effects)\n            sorted_effects = effects[idx]\n            n = len(sorted_effects)\n            z = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)\n            fig_daniel = go.Figure()\n            fig_daniel.add_trace(go.Scatter(x=sorted_effects, y=z, mode=\'markers+text\', text=[names[i] for i in idx], textposition="top center"))\n            mask = np.abs(sorted_effects) < limite\n            slope, intercept = np.polyfit(sorted_effects[mask], z[mask], 1) if sum(mask) > 1 else np.polyfit(sorted_effects, z, 1)\n            fig_daniel.add_trace(go.Scatter(x=[min(sorted_effects), max(sorted_effects)], y=[min(sorted_effects) * slope + intercept, max(sorted_effects) * slope + intercept], mode=\'lines\', line_color=\'blue\'))\n            fig_daniel.update_layout(xaxis_title="Efectos Estandarizados", yaxis_title="Proporción", height=400, showlegend=False)\n            st.plotly_chart(fig_daniel, use_container_width=True)\n    except Exception as e:\n        st.warning(f"No se pudieron generar los gráficos: {e}")\n\ndef ui_excluir_efectos(df, resp, anova, tipo_anova, key_suffix):\n    st.markdown("---")\n    st.subheader("🛠️ Refinamiento de Efectos (Excluir No Significativos)")\n    todos_efectos = [idx for idx in anova.index if idx != \'Residual\']\n    efectos_seleccionados = st.multiselect("Efectos a MANTENER en el modelo final:", todos_efectos, default=todos_efectos, key=f"excluir_efectos_{key_suffix}")\n\n    if st.button("🔄 Recalcular Modelo Reducido", key=f"btn_recalcular_{key_suffix}"):\n        if len(efectos_seleccionados) == 0:\n            st.warning("⚠️ Debes mantener al menos un efecto.")\n        else:\n            formula_reducida = f"{resp} ~ " + " + ".join(efectos_seleccionados)\n            analizar_anova_mejorada(df, formula_reducida, "Modelo Reducido (Mejor ANOVA)", f"{key_suffix}_red", "reducido", tipo_anova)\n            st.rerun()\n\n# ============================================================\n# CARGA DE DATOS, DISEÑO Y CONCLUSIONES\n# ============================================================\n\ndef crear_diseno_completo(factores_dict, respuestas_dict, n_replicas=1, randomize=True):\n    try:\n        niveles_factores = [factores_dict[f][\'niveles\'] for f in factores_dict]\n        nombres_factores = list(factores_dict.keys())\n        combinaciones = list(product(*niveles_factores))\n        df = pd.DataFrame(combinaciones, columns=nombres_factores)\n        if n_replicas > 1: df = pd.concat([df] * n_replicas, ignore_index=True)\n        if randomize: df = df.sample(frac=1, random_state=42).reset_index(drop=True)\n        for resp_name, resp_data in respuestas_dict.items():\n            df[resp_name] = np.nan\n        return df\n    except Exception as e:\n        st.error(f"❌ Error: {str(e)}")\n        return None\n\ndef generar_conclusiones_anova(anova_table, modelo, df, factor, respuesta, alpha=0.05):\n    conclusiones = []\n    p_valor = None\n    for col in anova_table.columns:\n        if \'P-Value\' in col: p_valor = anova_table[col].iloc[0]; break\n    if p_valor is None: return ["⚠️ No se obtuvo p-valor"], None\n    if p_valor < alpha: conclusiones.append(f"✅ **Conclusión:** Diferencia significativa (p = {p_valor:.6f})")\n    else: conclusiones.append(f"❌ **Conclusión:** Sin diferencia significativa (p = {p_valor:.6f})")\n    medias = df.groupby(factor)[respuesta].agg([\'mean\', \'std\', \'count\']).reset_index()\n    medias.columns = [factor, \'Media\', \'Desv\', \'n\']\n    medias_ordenadas = medias.sort_values(\'Media\', ascending=False)\n    for _, row in medias_ordenadas.iterrows():\n        conclusiones.append(f"  • {row[factor]}: Media = {row[\'Media\']:.4f} ± {row[\'Desv\']:.4f} (n={int(row[\'n\'])})")\n    mejor, peor = medias_ordenadas.iloc[0], medias_ordenadas.iloc[-1]\n    conclusiones.append(f"🏆 **Mejor:** {mejor[factor]} (Media = {mejor[\'Media\']:.4f})")\n    conclusiones.append(f"📉 **Peor:** {peor[factor]} (Media = {peor[\'Media\']:.4f})")\n    cv = (df[respuesta].std() / df[respuesta].mean()) * 100\n    conclusiones.append(f"📈 **CV:** {cv:.2f}%")\n    conclusiones.append(f"📐 **R² = {modelo.rsquared:.6f}**")\n    return conclusiones, medias_ordenadas\n\ndef generar_conclusiones_factorial(anova_table, modelo, df, factores, respuesta, alpha=0.05):\n    conclusiones = []\n    for i, row in anova_table.iterrows():\n        p_valor = None\n        for col in anova_table.columns:\n            if \'P-Value\' in col: p_valor = row[col]; break\n        if p_valor is None: continue\n        if p_valor < alpha: conclusiones.append(f"✅ **{i}:** Efecto significativo (p = {p_valor:.6f})")\n        else: conclusiones.append(f"❌ **{i}:** No significativo (p = {p_valor:.6f})")\n    conclusiones.append(f"📐 **R² = {modelo.rsquared:.6f}**")\n    return conclusiones\n\ndef detectar_separador(texto):\n    lineas = [l.strip() for l in texto.split(\'\\n\') if l.strip()]\n    if not lineas: return \',\'\n    conteos = {sep: sum(l.count(sep) for l in lineas) for sep in [\',\', \';\', \'\\t\', \'|\', \' \']}\n    mejor_sep = max(conteos, key=conteos.get)\n    return mejor_sep if conteos[mejor_sep] > 0 else \',\'\n\ndef limpiar_nombres_columnas(df):\n    df.columns = [c.strip().replace(\' \', \'_\').replace(\';\', \'\').replace(\',\', \'\') for c in df.columns]\n    return df\n\ndef convertir_tipos(df):\n    for col in df.columns:\n        try: df[col] = pd.to_numeric(df[col])\n        except: df[col] = df[col].astype(\'category\')\n    return df\n\ndef cargar_datos_desde_csv(archivo, separador=None):\n    try:\n        contenido = archivo.getvalue().decode(\'utf-8\')\n        if separador is None: separador = detectar_separador(contenido)\n        try: df = pd.read_csv(io.StringIO(contenido), sep=separador)\n        except:\n            for sep in [\',\', \';\', \'\\t\', \'|\']:\n                try:\n                    df = pd.read_csv(io.StringIO(contenido), sep=sep)\n                    if len(df.columns) > 1: break\n                except: continue\n        return convertir_tipos(limpiar_nombres_columnas(df))\n    except Exception as e:\n        st.error(f"❌ Error: {str(e)}")\n        return None\n\ndef cargar_datos_desde_url(url, separador=None):\n    try:\n        response = requests.get(url, timeout=30)\n        response.raise_for_status()\n        content = response.text\n        if separador is None: separador = detectar_separador(content)\n        try: df = pd.read_csv(io.StringIO(content), sep=separador)\n        except:\n            for sep in [\',\', \';\', \'\\t\', \'|\']:\n                try:\n                    df = pd.read_csv(io.StringIO(content), sep=sep)\n                    if len(df.columns) > 1: break\n                except: continue\n        return convertir_tipos(limpiar_nombres_columnas(df))\n    except Exception as e:\n        st.error(f"❌ Error: {str(e)}")\n        return None\n\ndef procesar_datos_texto(texto, n_columnas=None):\n    try:\n        separador = detectar_separador(texto)\n        lineas = [l.strip() for l in texto.split(\'\\n\') if l.strip()]\n        if not lineas: return None\n        valores = lineas[0].split(separador)\n        tiene_encabezado = False\n        try:\n            for v in valores: float(v.strip())\n        except: tiene_encabezado = True\n        datos, columnas = [], None\n        if tiene_encabezado:\n            columnas = [v.strip() for v in valores]\n            lineas = lineas[1:]\n        for linea in lineas:\n            valores = [v.strip() for v in linea.split(separador) if v.strip()]\n            if n_columnas is None or len(valores) == n_columnas:\n                try: datos.append([float(v) for v in valores])\n                except: datos.append(valores)\n        if not datos: return None\n        df = pd.DataFrame(datos)\n        if columnas and len(columnas) == len(df.columns): df.columns = columnas\n        else: df.columns = [f\'V{i+1}\' for i in range(len(df.columns))]\n        return convertir_tipos(limpiar_nombres_columnas(df))\n    except: return None\n\ndef obtener_datos_generales(key_suffix, n_columnas_esperadas=None):\n    st.subheader("📥 Carga de Datos")\n    tipo_carga = st.radio("Método:", ["✏️ Manual", "📁 CSV", "🔗 URL"], horizontal=True, key=f"tipo_carga_{key_suffix}")\n    df_resultado = None\n    if tipo_carga == "✏️ Manual":\n        if n_columnas_esperadas: st.caption(f"Formato: {n_columnas_esperadas} columnas")\n        datos_texto = st.text_area("Datos:", value="", height=200, key=f"text_area_{key_suffix}", placeholder="A,23\\nA,25\\nB,30")\n        if datos_texto and st.button("📊 Procesar", key=f"procesar_text_{key_suffix}"):\n            df_resultado = procesar_datos_texto(datos_texto, n_columnas_esperadas)\n            if df_resultado is not None:\n                st.success(f"✅ {len(df_resultado)} filas cargadas")\n                st.dataframe(df_resultado.head())\n                st.session_state.datos_cargados[key_suffix] = df_resultado\n    elif tipo_carga == "📁 CSV":\n        archivo = st.file_uploader("Archivo:", type=[\'csv\', \'txt\'], key=f"csv_upload_{key_suffix}")\n        if archivo is not None and st.button("📊 Cargar", key=f"cargar_csv_{key_suffix}"):\n            df_resultado = cargar_datos_desde_csv(archivo)\n            if df_resultado is not None:\n                st.success(f"✅ {len(df_resultado)} filas")\n                st.dataframe(df_resultado.head())\n                st.session_state.datos_cargados[key_suffix] = df_resultado\n    elif tipo_carga == "🔗 URL":\n        url = st.text_input("URL:", key=f"url_input_{key_suffix}")\n        if url and st.button("📊 Cargar", key=f"cargar_url_{key_suffix}"):\n            df_resultado = cargar_datos_desde_url(url)\n            if df_resultado is not None:\n                st.success(f"✅ {len(df_resultado)} filas")\n                st.dataframe(df_resultado.head())\n                st.session_state.datos_cargados[key_suffix] = df_resultado\n    if key_suffix in st.session_state.datos_cargados:\n        df_guardado = st.session_state.datos_cargados[key_suffix]\n        st.info(f"📊 Datos previos: {len(df_guardado)} filas")\n        return df_guardado\n    return df_resultado\n\ndef mostrar_informacion_datos(df, key_suffix):\n    if df is not None:\n        col1, col2 = st.columns(2)\n        with col1: st.metric("Filas", len(df))\n        with col2: st.metric("Columnas", len(df.columns))\n\n# ============================================================\n\n# ============================================================\n# INTERFAZ PRINCIPAL — CALCULADORA DE DISEÑOS EXPERIMENTALES\n# La pestaña Diseño Interactivo vive ahora en una aplicación independiente.\n# ============================================================\n\ntab1, tab2, tab3, tab4, tab5, tab6, tab6_resultados, tab7, tab8, tab9, tab10, tab11, tab12, tab13 = st.tabs([\n    "📊 ANOVA Un Factor", "📊 Diseño en Bloques",\n    "📐 Cuadrado Latino", "📐 Greco-Latino", "🎯 Factorial Dos Factores",\n    "🎯 Factorial Tres Factores", "🔬 Pruebas F2³", "📊 Diseño 2^k",\n    "📊 Fraccionado", "📈 MSR", "⚙️ Taguchi", "🎯 Optimización",\n    "➕ Aditivos", "📦 Split-Plot"\n])\n\n# PESTAÑAS 1 A 7 Y 9 A 13 (RESTAURADAS EXACTAMENTE IGUAL)\n# ============================================================\nwith tab1:\n    col1, col2 = st.columns([1, 2])\n    with col1:\n        st.subheader("📌 Diseño Completamente Aleatorizado")\n        df_temp = obtener_datos_generales("dca", 2)\n        if df_temp is not None and len(df_temp.columns) >= 2:\n            tipo_anova = selector_tipo_anova("dca")\n            col_factor = st.selectbox("Selecciona la columna del factor:", df_temp.columns.tolist(), key="factor_dca")\n            col_respuesta = st.selectbox("Selecciona la columna de respuesta:", [col for col in df_temp.columns if col != col_factor], key="respuesta_dca")\n            if st.button("🔍 Calcular ANOVA", key="dca_btn"):\n                df = df_temp.copy()\n                df.columns = [\'Tratamiento\' if col == col_factor else \'Valor\' if col == col_respuesta else col for col in df.columns]\n                df = df[[\'Tratamiento\', \'Valor\']]; df[\'Tratamiento\'] = df[\'Tratamiento\'].astype(\'category\')\n                modelo, anova = analizar_anova_mejorada(df, \'Valor ~ C(Tratamiento)\', "ANOVA Un Factor", "dca", "dca", tipo_anova)\n                if modelo is not None:\n                    st.session_state.pruebas_multiples[\'dca\'] = {\'df\': df, \'modelo\': modelo, \'anova\': anova}\n                    fig_sc = graficar_sumas_cuadrados(anova)\n                    if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n                    mostrar_graficos_residuos_directo(modelo, df, \'Tratamiento\', \'Valor\', "dca", "Gráficos de Residuos - DCA")\n    with col2:\n        if \'dca\' in st.session_state.resultados_anova:\n            res = st.session_state.resultados_anova[\'dca\']\n            df, modelo, anova = res[\'df\'], res[\'modelo\'], res[\'anova\']\n            mostrar_summary_statistics(modelo, df, [\'Tratamiento\'], \'Valor\', "dca")\n            fig = px.box(df, x=\'Tratamiento\', y=\'Valor\', color=\'Tratamiento\', title="Distribución por Tratamiento"); fig.update_layout(showlegend=False, height=350); st.plotly_chart(fig, use_container_width=True)\n            medias = df.groupby(\'Tratamiento\')[\'Valor\'].mean().reset_index()\n            fig2 = px.bar(medias, x=\'Tratamiento\', y=\'Valor\', title="Medias por Tratamiento", color=\'Tratamiento\', text_auto=True); fig2.update_layout(showlegend=False, height=300); fig2.update_traces(texttemplate=\'%{text:.3f}\', textposition=\'outside\'); st.plotly_chart(fig2, use_container_width=True)\n            st.subheader("🔬 Pruebas Múltiples")\n            with st.form(key="form_pruebas_dca"):\n                metodo = st.radio("Selecciona el método:", ["Tukey HSD", "LSD (Diferencia Mínima Significativa)"], horizontal=True, key="metodo_dca_form")\n                submitted = st.form_submit_button("Calcular Prueba")\n                if submitted:\n                    mse_c, gl_c = extraer_mse_gl(anova)\n                    if metodo == "Tukey HSD":\n                        tukey = realizar_tukey(df, \'Tratamiento\', \'Valor\')\n                        if tukey is not None: st.session_state.pruebas_multiples[\'dca_resultado\'] = (\'tukey\', tukey)\n                    else:\n                        lsd_df = realizar_lsd(df, \'Tratamiento\', \'Valor\', mse_c, gl_c)\n                        if lsd_df is not None: st.session_state.pruebas_multiples[\'dca_resultado\'] = (\'lsd\', lsd_df, mse_c, gl_c)\n            if \'dca_resultado\' in st.session_state.pruebas_multiples:\n                res = st.session_state.pruebas_multiples[\'dca_resultado\']\n                if res[0] == \'tukey\':\n                    st.dataframe(pd.DataFrame(data=res[1].summary().data[1:], columns=res[1].summary().data[0]))\n                    for l in interpretar_tukey(res[1]): st.write(l)\n                else:\n                    st.write(f"**MSE = {res[2]:.4f}, GL Error = {int(res[3])}**"); st.dataframe(res[1])\n                    for l in interpretar_lsd(res[1]): st.write(l)\n            st.subheader("📝 Conclusiones del Análisis")\n            for c in generar_conclusiones_anova(anova, modelo, df, \'Tratamiento\', \'Valor\')[0]:\n                if c.startswith("✅") or c.startswith("🏆") or c.startswith("📈"): st.success(c)\n                elif c.startswith("❌"): st.error(c)\n                else: st.info(c)\n            if st.button("🗑️ Limpiar resultados", key="limpiar_dca"):\n                if \'dca\' in st.session_state.resultados_anova: del st.session_state.resultados_anova[\'dca\']\n                if \'dca_resultado\' in st.session_state.pruebas_multiples: del st.session_state.pruebas_multiples[\'dca_resultado\']\n                st.rerun()\n\nwith tab2:\n    col1, col2 = st.columns([1, 2])\n    with col1:\n        st.subheader("📌 Diseño en Bloques")\n        df_temp = obtener_datos_generales("dbca", 3)\n        if df_temp is not None and len(df_temp.columns) >= 3:\n            tipo_anova = selector_tipo_anova("dbca")\n            col_factor = st.selectbox("Selecciona la columna del factor:", df_temp.columns.tolist(), key="factor_dbca")\n            col_bloque = st.selectbox("Selecciona la columna del bloque:", [col for col in df_temp.columns if col != col_factor], key="bloque_dbca")\n            col_respuesta = st.selectbox("Selecciona la columna de respuesta:", [col for col in df_temp.columns if col not in [col_factor, col_bloque]], key="respuesta_dbca")\n            if st.button("🔍 Calcular", key="dbca_btn"):\n                df = df_temp.copy()\n                df.columns = [\'Tratamiento\' if col == col_factor else \'Bloque\' if col == col_bloque else \'Valor\' if col == col_respuesta else col for col in df.columns]\n                df = df[[\'Tratamiento\', \'Bloque\', \'Valor\']]; df[\'Tratamiento\'] = df[\'Tratamiento\'].astype(\'category\'); df[\'Bloque\'] = df[\'Bloque\'].astype(\'category\')\n                modelo, anova = analizar_anova_mejorada(df, \'Valor ~ C(Tratamiento) + C(Bloque)\', "ANOVA en Bloques", "dbca", "dbca", tipo_anova)\n                if modelo is not None:\n                    st.session_state.pruebas_multiples[\'dbca\'] = {\'df\': df, \'modelo\': modelo, \'anova\': anova}\n                    fig_sc = graficar_sumas_cuadrados(anova)\n                    if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n                    mostrar_graficos_residuos_directo(modelo, df, \'Tratamiento\', \'Valor\', "dbca", "Gráficos de Residuos - DBCA")\n    with col2:\n        if \'dbca\' in st.session_state.resultados_anova:\n            res = st.session_state.resultados_anova[\'dbca\']\n            df, modelo, anova = res[\'df\'], res[\'modelo\'], res[\'anova\']\n            mostrar_summary_statistics(modelo, df, [\'Tratamiento\', \'Bloque\'], \'Valor\', "dbca")\n            fig_box_trat = px.box(df, x=\'Tratamiento\', y=\'Valor\', color=\'Tratamiento\', title="Distribución por Tratamiento"); fig_box_trat.update_layout(showlegend=False, height=350); st.plotly_chart(fig_box_trat, use_container_width=True)\n            fig_box_bloque = px.box(df, x=\'Bloque\', y=\'Valor\', color=\'Bloque\', title="Distribución por Bloque"); fig_box_bloque.update_layout(showlegend=False, height=350); st.plotly_chart(fig_box_bloque, use_container_width=True)\n            fig_facet = px.box(df, x=\'Tratamiento\', y=\'Valor\', color=\'Tratamiento\', facet_col=\'Bloque\'); fig_facet.update_layout(height=350); st.plotly_chart(fig_facet, use_container_width=True)\n            medias = df.groupby(\'Tratamiento\')[\'Valor\'].mean().reset_index()\n            fig2 = px.bar(medias, x=\'Tratamiento\', y=\'Valor\', title="Medias por Tratamiento", color=\'Tratamiento\', text_auto=True); fig2.update_layout(showlegend=False, height=300); fig2.update_traces(texttemplate=\'%{text:.3f}\', textposition=\'outside\'); st.plotly_chart(fig2, use_container_width=True)\n            st.subheader("🔬 Pruebas Múltiples para Tratamientos")\n            with st.form(key="form_pruebas_dbca"):\n                metodo = st.radio("Selecciona el método:", ["Tukey HSD", "LSD (Diferencia Mínima Significativa)"], horizontal=True, key="metodo_dbca_form")\n                submitted = st.form_submit_button("Calcular Prueba")\n                if submitted:\n                    mse_c, gl_c = extraer_mse_gl(anova)\n                    if metodo == "Tukey HSD":\n                        tukey = realizar_tukey(df, \'Tratamiento\', \'Valor\')\n                        if tukey is not None: st.session_state.pruebas_multiples[\'dbca_resultado\'] = (\'tukey\', tukey)\n                    else:\n                        lsd_df = realizar_lsd(df, \'Tratamiento\', \'Valor\', mse_c, gl_c)\n                        if lsd_df is not None: st.session_state.pruebas_multiples[\'dbca_resultado\'] = (\'lsd\', lsd_df, mse_c, gl_c)\n            if \'dbca_resultado\' in st.session_state.pruebas_multiples:\n                res = st.session_state.pruebas_multiples[\'dbca_resultado\']\n                if res[0] == \'tukey\':\n                    st.dataframe(pd.DataFrame(data=res[1].summary().data[1:], columns=res[1].summary().data[0]))\n                    for l in interpretar_tukey(res[1]): st.write(l)\n                else:\n                    st.write(f"**MSE = {res[2]:.4f}, GL Error = {int(res[3])}**"); st.dataframe(res[1])\n                    for l in interpretar_lsd(res[1]): st.write(l)\n            st.subheader("📝 Conclusiones del Análisis")\n            for c in generar_conclusiones_anova(anova, modelo, df, \'Tratamiento\', \'Valor\')[0]:\n                if c.startswith("✅") or c.startswith("🏆") or c.startswith("📈"): st.success(c)\n                elif c.startswith("❌"): st.error(c)\n                else: st.info(c)\n            if st.button("🗑️ Limpiar resultados", key="limpiar_dbca"):\n                if \'dbca\' in st.session_state.resultados_anova: del st.session_state.resultados_anova[\'dbca\']\n                if \'dbca_resultado\' in st.session_state.pruebas_multiples: del st.session_state.pruebas_multiples[\'dbca_resultado\']\n                st.rerun()\n\nwith tab3:\n    col1, col2 = st.columns([1, 2])\n    with col1:\n        st.subheader("📌 Cuadrado Latino")\n        df_temp = obtener_datos_generales("cl", 4)\n        if df_temp is not None and len(df_temp.columns) >= 4:\n            tipo_anova = selector_tipo_anova("cl")\n            col_fila = st.selectbox("Columna de fila:", df_temp.columns.tolist(), key="fila_cl")\n            cols_restantes = [c for c in df_temp.columns if c != col_fila]\n            col_columna = st.selectbox("Columna de columna:", cols_restantes, key="columna_cl")\n            cols_restantes = [c for c in cols_restantes if c != col_columna]\n            col_tratamiento = st.selectbox("Columna de tratamiento:", cols_restantes, key="tratamiento_cl")\n            cols_restantes = [c for c in cols_restantes if c != col_tratamiento]\n            col_respuesta = st.selectbox("Columna de respuesta:", cols_restantes, key="respuesta_cl")\n            if st.button("🔍 Calcular", key="cl_btn"):\n                df = df_temp.copy()\n                df.columns = [\'Fila\' if col == col_fila else \'Columna\' if col == col_columna else \'Tratamiento\' if col == col_tratamiento else \'Valor\' if col == col_respuesta else col for col in df.columns]\n                df = df[[\'Fila\', \'Columna\', \'Tratamiento\', \'Valor\']]\n                for col in [\'Fila\', \'Columna\', \'Tratamiento\']: df[col] = df[col].astype(\'category\')\n                modelo, anova = analizar_anova_mejorada(df, \'Valor ~ C(Fila) + C(Columna) + C(Tratamiento)\', "Cuadrado Latino", "cl", "cl", tipo_anova)\n                if modelo is not None:\n                    st.session_state.pruebas_multiples[\'cl\'] = {\'df\': df, \'modelo\': modelo, \'anova\': anova}\n                    fig_sc = graficar_sumas_cuadrados(anova)\n                    if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n                    mostrar_graficos_residuos_directo(modelo, df, \'Tratamiento\', \'Valor\', "cl", "Gráficos de Residuos - Cuadrado Latino")\n    with col2:\n        if \'cl\' in st.session_state.resultados_anova:\n            res = st.session_state.resultados_anova[\'cl\']\n            df, modelo, anova = res[\'df\'], res[\'modelo\'], res[\'anova\']\n            mostrar_summary_statistics(modelo, df, [\'Fila\', \'Columna\', \'Tratamiento\'], \'Valor\', "cl")\n            fig = px.box(df, x=\'Tratamiento\', y=\'Valor\', color=\'Tratamiento\', title="Distribución por Tratamiento"); fig.update_layout(showlegend=False, height=350); st.plotly_chart(fig, use_container_width=True)\n            medias = df.groupby(\'Tratamiento\')[\'Valor\'].mean().reset_index()\n            fig2 = px.bar(medias, x=\'Tratamiento\', y=\'Valor\', title="Medias por Tratamiento", color=\'Tratamiento\', text_auto=True); fig2.update_layout(showlegend=False, height=300); fig2.update_traces(texttemplate=\'%{text:.3f}\', textposition=\'outside\'); st.plotly_chart(fig2, use_container_width=True)\n            pivot = df.pivot(index=\'Fila\', columns=\'Columna\', values=\'Valor\')\n            fig3 = px.imshow(pivot, text_auto=True, aspect="auto", color_continuous_scale=\'Blues\', title="Mapa de Calor: Fila × Columna"); fig3.update_layout(height=350); st.plotly_chart(fig3, use_container_width=True)\n            st.subheader("🔬 Pruebas Múltiples para Tratamientos")\n            with st.form(key="form_pruebas_cl"):\n                metodo = st.radio("Selecciona el método:", ["Tukey HSD", "LSD (Diferencia Mínima Significativa)"], horizontal=True, key="metodo_cl_form")\n                submitted = st.form_submit_button("Calcular Prueba")\n                if submitted:\n                    mse_c, gl_c = extraer_mse_gl(anova)\n                    if metodo == "Tukey HSD":\n                        tukey = realizar_tukey(df, \'Tratamiento\', \'Valor\')\n                        if tukey is not None: st.session_state.pruebas_multiples[\'cl_resultado\'] = (\'tukey\', tukey)\n                    else:\n                        lsd_df = realizar_lsd(df, \'Tratamiento\', \'Valor\', mse_c, gl_c)\n                        if lsd_df is not None: st.session_state.pruebas_multiples[\'cl_resultado\'] = (\'lsd\', lsd_df, mse_c, gl_c)\n            if \'cl_resultado\' in st.session_state.pruebas_multiples:\n                res = st.session_state.pruebas_multiples[\'cl_resultado\']\n                if res[0] == \'tukey\':\n                    st.dataframe(pd.DataFrame(data=res[1].summary().data[1:], columns=res[1].summary().data[0]))\n                    for l in interpretar_tukey(res[1]): st.write(l)\n                else:\n                    st.write(f"**MSE = {res[2]:.4f}, GL Error = {int(res[3])}**"); st.dataframe(res[1])\n                    for l in interpretar_lsd(res[1]): st.write(l)\n            st.subheader("📝 Conclusiones del Análisis")\n            for c in generar_conclusiones_anova(anova, modelo, df, \'Tratamiento\', \'Valor\')[0]:\n                if c.startswith("✅") or c.startswith("🏆") or c.startswith("📈"): st.success(c)\n                elif c.startswith("❌"): st.error(c)\n                else: st.info(c)\n            if st.button("🗑️ Limpiar resultados", key="limpiar_cl"):\n                if \'cl\' in st.session_state.resultados_anova: del st.session_state.resultados_anova[\'cl\']\n                if \'cl_resultado\' in st.session_state.pruebas_multiples: del st.session_state.pruebas_multiples[\'cl_resultado\']\n                st.rerun()\n\nwith tab4:\n    col1, col2 = st.columns([1, 2])\n    with col1:\n        st.subheader("📌 Cuadrado Greco-Latino")\n        df_temp = obtener_datos_generales("cgl", 5)\n        if df_temp is not None and len(df_temp.columns) >= 5:\n            tipo_anova = selector_tipo_anova("cgl")\n            cols_disponibles = df_temp.columns.tolist()\n            col_fila = st.selectbox("Columna de fila:", cols_disponibles, key="fila_cgl")\n            cols_restantes = [c for c in cols_disponibles if c != col_fila]\n            col_columna = st.selectbox("Columna de columna:", cols_restantes, key="columna_cgl")\n            cols_restantes = [c for c in cols_restantes if c != col_columna]\n            col_tratamiento = st.selectbox("Columna de tratamiento:", cols_restantes, key="tratamiento_cgl")\n            cols_restantes = [c for c in cols_restantes if c != col_tratamiento]\n            col_greco = st.selectbox("Columna de greco:", cols_restantes, key="greco_cgl")\n            cols_restantes = [c for c in cols_restantes if c != col_greco]\n            col_respuesta = st.selectbox("Columna de respuesta:", cols_restantes, key="respuesta_cgl")\n            if st.button("🔍 Calcular", key="cgl_btn"):\n                df = df_temp.copy()\n                df.columns = [\'Fila\' if col == col_fila else \'Columna\' if col == col_columna else \'Tratamiento\' if col == col_tratamiento else \'Greco\' if col == col_greco else \'Valor\' if col == col_respuesta else col for col in df.columns]\n                df = df[[\'Fila\', \'Columna\', \'Tratamiento\', \'Greco\', \'Valor\']]\n                for col in [\'Fila\', \'Columna\', \'Tratamiento\', \'Greco\']: df[col] = df[col].astype(\'category\')\n                modelo, anova = analizar_anova_mejorada(df, \'Valor ~ C(Fila) + C(Columna) + C(Tratamiento) + C(Greco)\', "Greco-Latino", "cgl", "cgl", tipo_anova)\n                if modelo is not None:\n                    st.session_state.pruebas_multiples[\'cgl\'] = {\'df\': df, \'modelo\': modelo, \'anova\': anova}\n                    fig_sc = graficar_sumas_cuadrados(anova)\n                    if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n                    mostrar_graficos_residuos_directo(modelo, df, \'Tratamiento\', \'Valor\', "cgl", "Gráficos de Residuos - Greco-Latino")\n    with col2:\n        if \'cgl\' in st.session_state.resultados_anova:\n            res = st.session_state.resultados_anova[\'cgl\']\n            df, modelo, anova = res[\'df\'], res[\'modelo\'], res[\'anova\']\n            mostrar_summary_statistics(modelo, df, [\'Fila\', \'Columna\', \'Tratamiento\', \'Greco\'], \'Valor\', "cgl")\n            fig = px.box(df, x=\'Tratamiento\', y=\'Valor\', color=\'Tratamiento\'); fig.update_layout(showlegend=False, height=350); st.plotly_chart(fig, use_container_width=True)\n            medias = df.groupby(\'Tratamiento\')[\'Valor\'].mean().reset_index()\n            fig2 = px.bar(medias, x=\'Tratamiento\', y=\'Valor\', title="Medias por Tratamiento", color=\'Tratamiento\', text_auto=True); fig2.update_layout(showlegend=False, height=300); fig2.update_traces(texttemplate=\'%{text:.3f}\', textposition=\'outside\'); st.plotly_chart(fig2, use_container_width=True)\n            pivot = df.pivot(index=\'Fila\', columns=\'Columna\', values=\'Valor\')\n            fig3 = px.imshow(pivot, text_auto=True, aspect="auto", color_continuous_scale=\'Greens\', title="Mapa de Calor: Fila × Columna"); fig3.update_layout(height=350); st.plotly_chart(fig3, use_container_width=True)\n            try:\n                pivot_tg = df.pivot_table(index=\'Tratamiento\', columns=\'Greco\', values=\'Valor\', aggfunc=\'mean\')\n                fig4 = px.imshow(pivot_tg, text_auto=True, aspect="auto", color_continuous_scale=\'Blues\', title="Mapa de Calor: Tratamiento × Greco"); fig4.update_layout(height=350); st.plotly_chart(fig4, use_container_width=True)\n            except: pass\n            st.subheader("🔬 Pruebas Múltiples - Tratamiento")\n            with st.form(key="form_pruebas_cgl_trat"):\n                metodo_trat = st.radio("Método:", ["Tukey HSD", "LSD (Diferencia Mínima Significativa)"], horizontal=True, key="metodo_cgl_trat_form")\n                submitted_trat = st.form_submit_button("Calcular Prueba para Tratamiento")\n                if submitted_trat:\n                    mse_c, gl_c = extraer_mse_gl(anova)\n                    if metodo_trat == "Tukey HSD":\n                        tukey = realizar_tukey(df, \'Tratamiento\', \'Valor\')\n                        if tukey is not None: st.session_state.pruebas_multiples[\'cgl_trat_resultado\'] = (\'tukey\', tukey)\n                    else:\n                        lsd_df = realizar_lsd(df, \'Tratamiento\', \'Valor\', mse_c, gl_c)\n                        if lsd_df is not None: st.session_state.pruebas_multiples[\'cgl_trat_resultado\'] = (\'lsd\', lsd_df, mse_c, gl_c)\n            if \'cgl_trat_resultado\' in st.session_state.pruebas_multiples:\n                res = st.session_state.pruebas_multiples[\'cgl_trat_resultado\']\n                if res[0] == \'tukey\':\n                    st.dataframe(pd.DataFrame(data=res[1].summary().data[1:], columns=res[1].summary().data[0]))\n                    for l in interpretar_tukey(res[1]): st.write(l)\n                else:\n                    st.write(f"**MSE = {res[2]:.4f}, GL Error = {int(res[3])}**"); st.dataframe(res[1])\n                    for l in interpretar_lsd(res[1]): st.write(l)\n            st.subheader("🔬 Pruebas Múltiples - Greco")\n            with st.form(key="form_pruebas_cgl_greco"):\n                metodo_greco = st.radio("Método:", ["Tukey HSD", "LSD (Diferencia Mínima Significativa)"], horizontal=True, key="metodo_cgl_greco_form")\n                submitted_greco = st.form_submit_button("Calcular Prueba para Greco")\n                if submitted_greco:\n                    mse_c, gl_c = extraer_mse_gl(anova)\n                    if metodo_greco == "Tukey HSD":\n                        tukey = realizar_tukey(df, \'Greco\', \'Valor\')\n                        if tukey is not None: st.session_state.pruebas_multiples[\'cgl_greco_resultado\'] = (\'tukey\', tukey)\n                    else:\n                        lsd_df = realizar_lsd(df, \'Greco\', \'Valor\', mse_c, gl_c)\n                        if lsd_df is not None: st.session_state.pruebas_multiples[\'cgl_greco_resultado\'] = (\'lsd\', lsd_df, mse_c, gl_c)\n            if \'cgl_greco_resultado\' in st.session_state.pruebas_multiples:\n                res = st.session_state.pruebas_multiples[\'cgl_greco_resultado\']\n                if res[0] == \'tukey\':\n                    st.dataframe(pd.DataFrame(data=res[1].summary().data[1:], columns=res[1].summary().data[0]))\n                    for l in interpretar_tukey(res[1]): st.write(l)\n                else:\n                    st.write(f"**MSE = {res[2]:.4f}, GL Error = {int(res[3])}**"); st.dataframe(res[1])\n                    for l in interpretar_lsd(res[1]): st.write(l)\n            st.subheader("📝 Conclusiones del Análisis")\n            for c in generar_conclusiones_anova(anova, modelo, df, \'Tratamiento\', \'Valor\')[0]:\n                if c.startswith("✅") or c.startswith("🏆") or c.startswith("📈"): st.success(c)\n                elif c.startswith("❌"): st.error(c)\n                else: st.info(c)\n            if st.button("🗑️ Limpiar resultados", key="limpiar_cgl"):\n                for key in [\'cgl\', \'cgl_trat_resultado\', \'cgl_greco_resultado\']:\n                    if key in st.session_state.resultados_anova: del st.session_state.resultados_anova[key]\n                    if key in st.session_state.pruebas_multiples: del st.session_state.pruebas_multiples[key]\n                st.rerun()\n\nwith tab5:\n    col1, col2 = st.columns([1, 2])\n    with col1:\n        st.subheader("📌 Diseño Factorial 2²")\n        df_temp = obtener_datos_generales("f22", 3)\n        if df_temp is not None and len(df_temp.columns) >= 3:\n            tipo_anova = selector_tipo_anova("f22")\n            col_a = st.selectbox("Factor A:", df_temp.columns.tolist(), key="a_f22")\n            cols_restantes = [c for c in df_temp.columns if c != col_a]\n            col_b = st.selectbox("Factor B:", cols_restantes, key="b_f22")\n            cols_restantes = [c for c in cols_restantes if c != col_b]\n            col_respuesta = st.selectbox("Respuesta (Y):", cols_restantes, key="respuesta_f22")\n            if st.button("🔍 Calcular", key="f22_btn"):\n                df = df_temp.copy()\n                df.columns = [\'A\' if col == col_a else \'B\' if col == col_b else \'Y\' if col == col_respuesta else col for col in df.columns]\n                df = df[[\'A\', \'B\', \'Y\']]; df[\'A\'] = df[\'A\'].astype(\'category\'); df[\'B\'] = df[\'B\'].astype(\'category\')\n                modelo, anova = analizar_anova_mejorada(df, \'Y ~ A * B\', "Factorial 2²", "f22", "f22", tipo_anova)\n                if modelo is not None:\n                    st.session_state.pruebas_multiples[\'f22\'] = {\'df\': df, \'modelo\': modelo, \'anova\': anova}\n                    fig_sc = graficar_sumas_cuadrados(anova)\n                    if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n                    mostrar_graficos_residuos_directo(modelo, df, \'A\', \'Y\', "f22", "Gráficos de Residuos - Factorial 2²")\n    with col2:\n        if \'f22\' in st.session_state.resultados_anova:\n            res = st.session_state.resultados_anova[\'f22\']\n            df, modelo, anova = res[\'df\'], res[\'modelo\'], res[\'anova\']\n            mostrar_summary_statistics(modelo, df, [\'A\', \'B\'], \'Y\', "f22")\n            fig_box = graficar_boxplot_por_factor(df, [\'A\', \'B\'], \'Y\')\n            if fig_box: st.plotly_chart(fig_box, use_container_width=True)\n            fig_inter = graficar_interaccion_mejorada(df, \'A\', \'B\', \'Y\')\n            if fig_inter: st.plotly_chart(fig_inter, use_container_width=True)\n            df[\'Combinacion\'] = df[\'A\'].astype(str) + \',\' + df[\'B\'].astype(str)\n            fig1 = px.box(df, x=\'Combinacion\', y=\'Y\', color=\'Combinacion\', title="Distribución por Combinación"); fig1.update_layout(showlegend=False, height=400); st.plotly_chart(fig1, use_container_width=True)\n            medias = df.groupby([\'A\', \'B\'])[\'Y\'].mean().reset_index()\n            fig3 = px.bar(medias, x=\'A\', y=\'Y\', color=\'B\', title="Medias por Combinación", barmode=\'group\', text_auto=True); fig3.update_layout(height=350); fig3.update_traces(texttemplate=\'%{text:.3f}\', textposition=\'outside\'); st.plotly_chart(fig3, use_container_width=True)\n            st.subheader("🔬 Pruebas Múltiples")\n            with st.form(key="form_pruebas_f22"):\n                metodo = st.radio("Método:", ["Tukey HSD", "LSD (Diferencia Mínima Significativa)"], horizontal=True, key="metodo_f22_form")\n                factores_prueba = st.multiselect("Factores para probar:", [\'A\', \'B\'], default=[\'A\', \'B\'], key="factores_f22_form")\n                submitted = st.form_submit_button("Calcular Pruebas")\n                if submitted:\n                    mse_c, gl_c = extraer_mse_gl(anova)\n                    resultados_multiples = {}\n                    for factor in factores_prueba:\n                        n_niveles = len(df[factor].unique())\n                        if metodo == "Tukey HSD":\n                            if n_niveles > 2:\n                                tukey = realizar_tukey(df, factor, \'Y\')\n                                if tukey is not None: resultados_multiples[factor] = (\'tukey\', tukey)\n                            else: st.info(f"ℹ️ **{factor}:** Tukey no aplica. Usa LSD.")\n                        else:\n                            lsd_df = realizar_lsd(df, factor, \'Y\', mse_c, gl_c)\n                            if lsd_df is not None: resultados_multiples[factor] = (\'lsd\', lsd_df, mse_c, gl_c)\n                    st.session_state.pruebas_multiples[\'f22_resultado\'] = resultados_multiples\n            if \'f22_resultado\' in st.session_state.pruebas_multiples:\n                for factor, r in st.session_state.pruebas_multiples[\'f22_resultado\'].items():\n                    st.markdown(f"#### 📌 Resultados para Factor {factor}")\n                    if r[0] == \'tukey\':\n                        st.dataframe(pd.DataFrame(data=r[1].summary().data[1:], columns=r[1].summary().data[0]))\n                        for l in interpretar_tukey(r[1]): st.write(l)\n                    else:\n                        st.write(f"**MSE = {r[2]:.4f}, GL Error = {int(r[3])}**"); st.dataframe(r[1])\n                        for l in interpretar_lsd(r[1]): st.write(l)\n            st.subheader("📝 Conclusiones del Análisis")\n            for c in generar_conclusiones_factorial(anova, modelo, df, [\'A\', \'B\'], \'Y\'):\n                if c.startswith("✅") or c.startswith("📈"): st.success(c)\n                elif c.startswith("❌"): st.error(c)\n                else: st.info(c)\n            if st.button("🗑️ Limpiar resultados", key="limpiar_f22"):\n                if \'f22\' in st.session_state.resultados_anova: del st.session_state.resultados_anova[\'f22\']\n                if \'f22_resultado\' in st.session_state.pruebas_multiples: del st.session_state.pruebas_multiples[\'f22_resultado\']\n                st.rerun()\n\nwith tab6:\n    col1, col2 = st.columns([1, 2])\n    with col1:\n        st.subheader("📌 Diseño Factorial 2³")\n        df_temp = obtener_datos_generales("f23", 4)\n        if df_temp is not None and len(df_temp.columns) >= 4:\n            tipo_anova = selector_tipo_anova("f23")\n            col_a = st.selectbox("Factor A:", df_temp.columns.tolist(), key="a_f23")\n            cols_restantes = [c for c in df_temp.columns if c != col_a]\n            col_b = st.selectbox("Factor B:", cols_restantes, key="b_f23")\n            cols_restantes = [c for c in cols_restantes if c != col_b]\n            col_c = st.selectbox("Factor C:", cols_restantes, key="c_f23")\n            cols_restantes = [c for c in cols_restantes if c != col_c]\n            col_respuesta = st.selectbox("Respuesta (Y):", cols_restantes, key="respuesta_f23")\n            if st.button("🔍 Calcular", key="f23_btn"):\n                df = df_temp.copy()\n                df.columns = [\'A\' if col == col_a else \'B\' if col == col_b else \'C\' if col == col_c else \'Y\' if col == col_respuesta else col for col in df.columns]\n                df = df[[\'A\', \'B\', \'C\', \'Y\']]\n                for col in [\'A\', \'B\', \'C\']: df[col] = df[col].astype(\'category\')\n                modelo, anova = analizar_anova_mejorada(df, \'Y ~ A * B * C\', "Factorial 2³", "f23", "f23", tipo_anova)\n                if modelo is not None:\n                    st.session_state.pruebas_multiples[\'f23\'] = {\'df\': df, \'modelo\': modelo, \'anova\': anova}\n                    fig_sc = graficar_sumas_cuadrados(anova)\n                    if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n                    st.subheader("📊 Gráficos de Interacción")\n                    factores_list = [\'A\', \'B\', \'C\']\n                    fig_interaccion = graficos_interaccion_multifactorial(df, factores_list, \'Y\', max_pares=3)\n                    if fig_interaccion is not None: st.plotly_chart(fig_interaccion, use_container_width=True)\n                    st.subheader("📊 Mapa de Calor de Interacciones")\n                    fig_heatmap = graficos_interaccion_heatmap(df, factores_list, \'Y\')\n                    if fig_heatmap is not None: st.plotly_chart(fig_heatmap, use_container_width=True)\n                    mostrar_graficos_residuos_directo(modelo, df, \'A\', \'Y\', "f23", "Gráficos de Residuos - Factorial 2³")\n    with col2:\n        if \'f23\' in st.session_state.resultados_anova:\n            res = st.session_state.resultados_anova[\'f23\']\n            df, modelo, anova = res[\'df\'], res[\'modelo\'], res[\'anova\']\n            mostrar_summary_statistics(modelo, df, [\'A\', \'B\', \'C\'], \'Y\', "f23")\n            fig_box = graficar_boxplot_por_factor(df, [\'A\', \'B\', \'C\'], \'Y\')\n            if fig_box: st.plotly_chart(fig_box, use_container_width=True)\n            fig_inter = graficar_interaccion_mejorada(df, \'A\', \'B\', \'Y\')\n            if fig_inter: st.plotly_chart(fig_inter, use_container_width=True)\n            df[\'Combinacion\'] = df[\'A\'].astype(str) + \',\' + df[\'B\'].astype(str) + \',\' + df[\'C\'].astype(str)\n            fig1 = px.box(df, x=\'Combinacion\', y=\'Y\', color=\'Combinacion\', title="Distribución por Combinación"); fig1.update_layout(showlegend=False, height=350); st.plotly_chart(fig1, use_container_width=True)\n            medias = df.groupby([\'A\', \'B\', \'C\'])[\'Y\'].mean().reset_index()\n            fig3 = px.bar(medias, x=\'A\', y=\'Y\', color=\'B\', facet_col=\'C\', title="Medias por Combinación", barmode=\'group\', text_auto=True); fig3.update_layout(height=350); fig3.update_traces(texttemplate=\'%{text:.3f}\', textposition=\'outside\'); st.plotly_chart(fig3, use_container_width=True)\n            st.subheader("📝 Conclusiones del Análisis")\n            for c in generar_conclusiones_factorial(anova, modelo, df, [\'A\', \'B\', \'C\'], \'Y\'):\n                if c.startswith("✅"): st.success(c)\n                elif c.startswith("❌"): st.error(c)\n                else: st.info(c)\n            if st.button("🗑️ Limpiar resultados", key="limpiar_f23"):\n                for key in [\'f23\', \'f23_resultado_tab\']:\n                    if key in st.session_state.resultados_anova: del st.session_state.resultados_anova[key]\n                    if key in st.session_state.pruebas_multiples: del st.session_state.pruebas_multiples[key]\n                st.rerun()\n\nwith tab6_resultados:\n    st.header("🔬 Pruebas Múltiples - Factorial 2³")\n    if \'f23\' in st.session_state.resultados_anova:\n        res = st.session_state.resultados_anova[\'f23\']\n        df, anova, modelo = res[\'df\'], res[\'anova\'], res[\'modelo\']\n        mse_c, gl_c = extraer_mse_gl(anova)\n        st.info(f"📊 **MSE del modelo completo:** {mse_c:.4f} | **GL Error:** {int(gl_c)}")\n        with st.expander("📋 Summary Statistics — Least Squares Means", expanded=False):\n            mostrar_summary_statistics(modelo, df, [\'A\', \'B\', \'C\'], \'Y\', "f23_resultados")\n        st.markdown("---")\n        st.subheader("⚙️ Configuración")\n        metodo = st.radio("Método:", ["Tukey HSD", "LSD (Diferencia Mínima Significativa)"], horizontal=True, key="met_f23_final")\n        col1, col2, col3 = st.columns(3)\n        with col1: usar_a = st.checkbox("Factor A", value=True, key="chk_a_f23_final")\n        with col2: usar_b = st.checkbox("Factor B", value=True, key="chk_b_f23_final")\n        with col3: usar_c = st.checkbox("Factor C", value=True, key="chk_c_f23_final")\n        factores_prueba = []\n        if usar_a: factores_prueba.append(\'A\')\n        if usar_b: factores_prueba.append(\'B\')\n        if usar_c: factores_prueba.append(\'C\')\n        if st.button("Calcular Pruebas Múltiples", key="btn_calc_f23_final"):\n            if not factores_prueba: st.warning("⚠️ Selecciona al menos un factor.")\n            else:\n                resultados_multiples = {}\n                for factor in factores_prueba:\n                    if metodo == "Tukey HSD":\n                        tukey = realizar_tukey(df, factor, \'Y\')\n                        if tukey is not None: resultados_multiples[factor] = (\'tukey\', tukey)\n                    else:\n                        lsd_df = realizar_lsd(df, factor, \'Y\', mse_c, gl_c)\n                        if lsd_df is not None: resultados_multiples[factor] = (\'lsd\', lsd_df, mse_c, gl_c)\n                st.session_state.pruebas_multiples[\'f23_resultado_tab\'] = resultados_multiples\n        if \'f23_resultado_tab\' in st.session_state.pruebas_multiples:\n            for factor, r in st.session_state.pruebas_multiples[\'f23_resultado_tab\'].items():\n                st.markdown(f"#### 📌 Resultados para Factor {factor}")\n                if r[0] == \'tukey\':\n                    st.dataframe(pd.DataFrame(data=r[1].summary().data[1:], columns=r[1].summary().data[0]))\n                    for l in interpretar_tukey(r[1]): st.write(l)\n                else:\n                    st.write(f"**MSE = {r[2]:.4f}, GL Error = {int(r[3])}**"); st.dataframe(r[1])\n                    for l in interpretar_lsd(r[1]): st.write(l)\n    else:\n        st.warning("⚠️ Primero calcula el ANOVA en la pestaña \'Factorial Tres Factores\'.")\n\n# ============================================================\n# TAB 7: DISEÑO 2^k (CON PARETO, DANIEL Y MODELO REDUCIDO)\n# ============================================================\nwith tab7:\n    st.subheader("📌 Diseño 2^k")\n    k = st.number_input("Factores (k):", min_value=2, max_value=8, value=3, step=1, key="k_2k")\n    df_temp = obtener_datos_generales("d2k", k+1)\n    if df_temp is not None and len(df_temp.columns) >= k+1:\n        tipo_anova = selector_tipo_anova("d2k")\n        factores = []\n        cols = df_temp.columns.tolist()\n        for i in range(k):\n            f = st.selectbox(f"Factor {i+1}:", cols if i == 0 else [c for c in cols if c not in factores], key=f"f_{i}_2k")\n            factores.append(f)\n        col_r = st.selectbox("Respuesta:", [c for c in cols if c not in factores], key="r_2k")\n        if st.button("🔍 Calcular", key="2k_btn"):\n            df = df_temp.copy()\n            mapeo = {factores[i]: f\'F{i+1}\' for i in range(k)}\n            mapeo[col_r] = \'Y\'\n            df.rename(columns=mapeo, inplace=True)\n            for i in range(k): df[f\'F{i+1}\'] = df[f\'F{i+1}\'].astype(\'category\')\n            factores_list = [f\'F{i+1}\' for i in range(k)]\n            modelo, anova = analizar_anova_mejorada(df, \'Y ~ \' + \' * \'.join(factores_list), f"2^{k}", "d2k", "d2k", tipo_anova)\n\n            if modelo is not None:\n                st.session_state.pruebas_multiples[\'d2k\'] = {\'df\': df, \'modelo\': modelo, \'anova\': anova}\n                mostrar_tabla_anova("d2k")\n                mostrar_summary_statistics(modelo, df, factores_list, \'Y\', "d2k")\n                fig_sc = graficar_sumas_cuadrados(anova)\n                if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n\n                # ⭐ NUEVO: Pareto, Daniel y Excluir Efectos\n                mostrar_pareto_y_daniel(modelo, anova, \'Y\', len(df))\n                ui_excluir_efectos(df, \'Y\', anova, tipo_anova, "d2k")\n\n                # ⭐ MOSTRAR MEJOR ANOVA\n                if \'d2k_red\' in st.session_state.resultados_anova:\n                    st.markdown("### 🌟 Mejor ANOVA (Modelo Reducido)")\n                    mostrar_tabla_anova("d2k_red")\n\n                mostrar_graficos_residuos_directo(modelo, df, \'F1\', \'Y\', "d2k", "Residuos")\n\n# ============================================================\n# ⭐ TAB 8: DISEÑO FRACCIONADO (CON TABLAS ANOVA Y MEJOR ANOVA)\n# ============================================================\ndef generar_diseno_fraccionado(k, p, randomize=True):\n    basicos = k - p\n    if basicos < 2: return None\n    combinaciones = list(product(*[[-1, 1]] * basicos))\n    df = pd.DataFrame(combinaciones, columns=[chr(65 + i) for i in range(basicos)])\n    generadores = {(3, 1): [\'AB\'], (4, 1): [\'ABC\'], (5, 1): [\'ABCD\'], (5, 2): [\'AB\', \'AC\'], (6, 1): [\'ABCDE\'], (6, 2): [\'ABC\', \'ABD\'], (7, 1): [\'ABCDEF\'], (7, 2): [\'ABCD\', \'ABCE\'], (7, 3): [\'ABC\', \'BCD\', \'ACD\'], (7, 4): [\'AB\', \'AC\', \'BC\', \'ABC\'], (8, 1): [\'ABCDEFG\'], (8, 2): [\'ABCD\', \'ABCE\'], (8, 3): [\'ABC\', \'BCD\', \'ACD\'], (8, 4): [\'AB\', \'AC\', \'BC\', \'ABC\']}\n    gen_list = generadores.get((k, p), [\'AB\', \'AC\', \'BC\'][:p])\n    for i, gen in enumerate(gen_list):\n        val = np.ones(len(df))\n        for char in gen:\n            if char in df.columns: val *= df[char]\n        df[chr(65 + basicos + i)] = val.astype(int)\n    if randomize: df = df.sample(frac=1, random_state=42).reset_index(drop=True)\n    return df\n\nwith tab8:\n    st.header("📌 Diseño Fraccionado 2^(k-p)")\n\n    st.subheader("⚙️ 1. Configuración de Datos")\n    origen = st.radio("Selecciona el origen de los datos:", ["Generar Diseño Vacío", "Cargar desde CSV", "Cargar desde URL"], horizontal=True, key="fr_origen")\n\n    if origen == "Generar Diseño Vacío":\n        col1, col2, col3, col4 = st.columns(4)\n        with col1: k = st.number_input("Factores (k):", min_value=3, max_value=8, value=7, step=1, key="fr_k")\n        with col2: p = st.number_input("Fracción (p):", min_value=1, max_value=max(1, k - 2), value=4, step=1, key="fr_p")\n        with col3: n_replicas = st.number_input("Réplicas:", min_value=1, max_value=5, value=1, step=1, key="fr_rep")\n        with col4: randomizar = st.checkbox("Aleatorizar", value=True, key="fr_rand")\n\n        resolucion = "III" if p >= 4 else "IV" if p >= 2 else "V"\n        n_base = 2 ** (k - p)\n        st.info(f"**Diseño 2^({k}-{p})** | Resolución: **{resolucion}** | Base de corridas: **{n_base}** | Total: **{n_base * n_replicas}**")\n\n        with st.expander("📝 Nombres de los Factores (opcional)", expanded=False):\n            nombres_factores = []\n            cols_nombres = st.columns(min(k, 5))\n            for i in range(k):\n                with cols_nombres[i % len(cols_nombres)]:\n                    nombres_factores.append(st.text_input(f"Factor {chr(65+i)}:", value=f"Factor_{chr(65+i)}", key=f"fr_nombre_{i}"))\n\n        if st.button("🎲 Generar Diseño", key="fr_btn_generar", type="primary"):\n            df_gen = generar_diseno_fraccionado(k, p, randomizar)\n            if df_gen is not None:\n                if n_replicas > 1: df_gen = pd.concat([df_gen] * n_replicas, ignore_index=True).reset_index(drop=True)\n                df_gen = df_gen.rename(columns={chr(65 + i): nombres_factores[i] for i in range(k)})\n                df_gen[\'Respuesta\'] = np.nan\n                st.session_state.fr_df = df_gen\n                st.session_state.fr_analizado = False\n                st.session_state.fr_colapsado = None\n                st.rerun()\n\n    elif origen == "Cargar desde CSV":\n        archivo = st.file_uploader("Sube tu archivo CSV (debe incluir la columna \'Respuesta\'):", type=[\'csv\', \'txt\'], key="fr_upload")\n        if archivo is not None and st.button("📊 Cargar CSV", key="fr_btn_csv"):\n            df_subido = cargar_datos_desde_csv(archivo)\n            if df_subido is not None:\n                st.session_state.fr_df = df_subido\n                st.session_state.fr_analizado = False\n                st.session_state.fr_colapsado = None\n                st.rerun()\n\n    elif origen == "Cargar desde URL":\n        url = st.text_input("URL del CSV:", key="fr_url")\n        if url and st.button("📊 Cargar URL", key="fr_btn_url"):\n            df_subido = cargar_datos_desde_url(url)\n            if df_subido is not None:\n                st.session_state.fr_df = df_subido\n                st.session_state.fr_analizado = False\n                st.session_state.fr_colapsado = None\n                st.rerun()\n\n    if st.button("🗑️ Limpiar Todo", key="fr_btn_limpiar"):\n        st.session_state.fr_df = None\n        st.session_state.fr_analizado = False\n        st.session_state.fr_colapsado = None\n        if \'fr\' in st.session_state.resultados_anova: del st.session_state.resultados_anova[\'fr\']\n        if \'fr_red\' in st.session_state.resultados_anova: del st.session_state.resultados_anova[\'fr_red\']\n        st.rerun()\n\n    if st.session_state.get(\'fr_df\') is not None:\n        st.markdown("---")\n        st.subheader("📋 Tabla del Diseño (edita la columna \'Respuesta\')")\n        df_editado = st.data_editor(st.session_state.fr_df, num_rows="fixed", key="fr_editor")\n        st.session_state.fr_df = df_editado\n\n        st.markdown("---")\n        st.subheader("📊 2. Análisis ANOVA y Gráficos")\n        tipo_anova = selector_tipo_anova("fr")\n        factores_cols = [c for c in df_editado.columns if c != \'Respuesta\' and c != \'Orden\']\n\n        if st.button("🔍 Calcular ANOVA Fraccionado", key="fr_btn_calc", type="primary"):\n            if df_editado[\'Respuesta\'].isnull().all():\n                st.error("❌ La columna \'Respuesta\' está vacía.")\n            else:\n                st.session_state.fr_analizado = True\n                st.session_state.fr_colapsado = None\n                df_analisis = df_editado.copy().reset_index(drop=True)\n                for f in factores_cols: df_analisis[f] = df_analisis[f].astype(\'category\')\n                formula = \'Respuesta ~ \' + \' + \'.join(factores_cols)\n                analizar_anova_mejorada(df_analisis, formula, "ANOVA Inicial (Efectos Principales)", "fr", "fr", tipo_anova)\n                st.rerun()\n\n        # Mostrar resultados persistentes\n        if st.session_state.fr_analizado and \'fr\' in st.session_state.resultados_anova:\n            res = st.session_state.resultados_anova[\'fr\']\n            modelo, anova, df_analisis = res[\'modelo\'], res[\'anova\'], res[\'df\']\n\n            # ⭐ 1. MOSTRAR TABLA ANOVA INICIAL\n            st.markdown("### 🔹 ANOVA Inicial")\n            mostrar_tabla_anova("fr")\n\n            mostrar_summary_statistics(modelo, df_analisis, factores_cols, \'Respuesta\', "fr")\n            mostrar_pareto_y_daniel(modelo, anova, \'Respuesta\', len(df_analisis))\n            graficar_efectos_e_interacciones(df_analisis, factores_cols, \'Respuesta\')\n\n            st.markdown("---")\n            st.subheader("🛠️ 3. Refinamiento (Selección del Mejor Modelo)")\n            todos_efectos = [idx for idx in anova.index if idx != \'Residual\']\n            efectos_seleccionados = st.multiselect("Efectos a MANTENER en el modelo final:", todos_efectos, default=todos_efectos, key="fr_excluir")\n\n            if st.button("🔄 Recalcular Modelo Reducido", key="fr_recalc"):\n                if len(efectos_seleccionados) == 0:\n                    st.warning("⚠️ Debes mantener al menos un efecto.")\n                else:\n                    formula_red = \'Respuesta ~ \' + \' + \'.join(efectos_seleccionados)\n                    analizar_anova_mejorada(df_analisis, formula_red, "Modelo Reducido (Mejor ANOVA)", "fr_red", "reducido", tipo_anova)\n                    st.rerun()\n\n            # ⭐ 2. MOSTRAR MEJOR ANOVA Y SUS RESULTADOS\n            if \'fr_red\' in st.session_state.resultados_anova:\n                res_red = st.session_state.resultados_anova[\'fr_red\']\n                st.markdown("### 🌟 Mejor ANOVA (Modelo Reducido)")\n                mostrar_tabla_anova("fr_red")\n                mostrar_summary_statistics(res_red[\'modelo\'], df_analisis, efectos_seleccionados, \'Respuesta\', "fr_red")\n                mostrar_graficos_residuos_directo(res_red[\'modelo\'], df_analisis, efectos_seleccionados[0], \'Respuesta\', "fr_red", "Residuos del Mejor Modelo")\n                st.subheader("📝 Conclusiones del Modelo Final")\n                for c in generar_conclusiones_factorial(res_red[\'anova\'], res_red[\'modelo\'], df_analisis, efectos_seleccionados, \'Respuesta\'):\n                    if c.startswith("✅"): st.success(c)\n                    elif c.startswith("❌"): st.error(c)\n                    else: st.info(c)\n\n            st.markdown("---")\n            st.subheader("📉 4. Colapsar Diseño")\n            col_fijar = st.selectbox("Factor a fijar:", factores_cols, key="fr_colapsar_factor")\n            niveles_fijar = sorted(df_analisis[col_fijar].dropna().unique(), key=lambda x: str(x))\n            col_nivel = st.selectbox("Nivel a fijar:", niveles_fijar, key="fr_colapsar_nivel")\n\n            if st.button("Colapsar y Re-analizar", key="fr_btn_colapsar"):\n                df_col = df_analisis[df_analisis[col_fijar] == col_nivel].copy().reset_index(drop=True)\n                f_restantes = [f for f in factores_cols if f != col_fijar]\n                if len(f_restantes) > 0 and not df_col[\'Respuesta\'].isnull().all():\n                    for f in f_restantes: df_col[f] = df_col[f].astype(\'category\')\n                    formula_col = \'Respuesta ~ \' + \' + \'.join(f_restantes)\n                    modelo_c, anova_c = analizar_anova_mejorada(df_col, formula_col, "ANOVA Colapsado", "fr_col", "colapsado", tipo_anova)\n                    if modelo_c is not None:\n                        st.session_state.fr_colapsado = (modelo_c, anova_c, df_col, f_restantes)\n                        st.rerun()\n\n            if st.session_state.fr_colapsado is not None:\n                modelo_c, anova_c, df_col, f_restantes = st.session_state.fr_colapsado\n                st.success(f"✅ Análisis colapsado (Fijando {col_fijar} en {col_nivel}):")\n                mostrar_tabla_anova("fr_col")\n                mostrar_summary_statistics(modelo_c, df_col, f_restantes, \'Respuesta\', "fr_col")\n                mostrar_pareto_y_daniel(modelo_c, anova_c, \'Respuesta\', len(df_col))\n                graficar_efectos_e_interacciones(df_col, f_restantes, \'Respuesta\')\n                if st.button("Cerrar Sub-análisis", key="fr_cerrar_colapso"):\n                    st.session_state.fr_colapsado = None\n                    st.rerun()\n    else:\n        st.info("👆 Selecciona un origen de datos y presiona el botón para comenzar.")\n\n# ============================================================\n# TAB 9 a 13\n# ============================================================\nwith tab9:\n    st.subheader("📌 MSR")\n    n_f = st.number_input("Factores:", min_value=2, max_value=4, value=2, step=1, key="nf_msr")\n    df_temp = obtener_datos_generales("msr", n_f+1)\n    if df_temp is not None and len(df_temp.columns) >= n_f+1:\n        tipo_anova = selector_tipo_anova("msr")\n        factores = []\n        cols = df_temp.columns.tolist()\n        for i in range(n_f):\n            f = st.selectbox(f"Factor {i+1}:", cols if i == 0 else [c for c in cols if c not in factores], key=f"f_{i}_msr")\n            factores.append(f)\n        col_r = st.selectbox("Respuesta:", [c for c in cols if c not in factores], key="r_msr")\n        if st.button("🔍 Calcular", key="msr_btn"):\n            df = df_temp.copy()\n            mapeo = {factores[i]: f\'F{i+1}\' for i in range(n_f)}\n            mapeo[col_r] = \'Y\'\n            df.rename(columns=mapeo, inplace=True)\n            vars_list = [f\'F{i+1}\' for i in range(n_f)]\n            formula = \'Y ~ \' + \' + \'.join(vars_list) + \' + \' + \' + \'.join([f\'I({v}**2)\' for v in vars_list])\n            if n_f >= 2: formula += \' + \' + \' + \'.join([f\'F{i+1}:F{j+1}\' for i in range(n_f) for j in range(i+1, n_f)])\n            modelo, anova = analizar_anova_mejorada(df, formula, "MSR", "msr", "msr", tipo_anova)\n            if modelo is not None:\n                mostrar_summary_statistics(modelo, df, vars_list, \'Y\', "msr")\n                mostrar_graficos_residuos_directo(modelo, df, \'F1\', \'Y\', "msr", "Residuos")\n\nwith tab10:\n    st.subheader("📌 Taguchi")\n    arreglo = st.selectbox("Arreglo:", ["L4 (2³)", "L8 (2⁷)", "L9 (3⁴)", "L16 (2¹⁵)", "L27 (3¹³)"], index=1, key="tag")\n    n_f = {"L4 (2³)": 3, "L8 (2⁷)": 7, "L9 (3⁴)": 4, "L16 (2¹⁵)": 15, "L27 (3¹³)": 13}[arreglo]\n    df_temp = obtener_datos_generales("tag", n_f+1)\n    if df_temp is not None and len(df_temp.columns) >= n_f+1:\n        tipo_anova = selector_tipo_anova("tag")\n        factores = []\n        cols = df_temp.columns.tolist()\n        for i in range(min(n_f, 8)):\n            f = st.selectbox(f"Factor {i+1}:", cols if i == 0 else [c for c in cols if c not in factores], key=f"f_{i}_tag")\n            factores.append(f)\n        col_r = st.selectbox("Respuesta:", [c for c in cols if c not in factores], key="r_tag")\n        if st.button("🔍 Calcular", key="tag_btn"):\n            df = df_temp.copy()\n            mapeo = {factores[i]: f\'F{i+1}\' for i in range(min(n_f, 8))}\n            restantes = [c for c in cols if c not in factores and c != col_r]\n            for i in range(8, n_f):\n                if restantes: mapeo[restantes.pop(0)] = f\'F{i+1}\'\n            mapeo[col_r] = \'Y\'\n            df.rename(columns=mapeo, inplace=True)\n            for i in range(n_f): df[f\'F{i+1}\'] = df[f\'F{i+1}\'].astype(\'category\')\n            factores_list = [f\'F{i+1}\' for i in range(n_f)]\n            modelo, anova = analizar_anova_mejorada(df, \'Y ~ \' + \' + \'.join(factores_list), "Taguchi", "tag", "tag", tipo_anova)\n            if modelo is not None:\n                mostrar_summary_statistics(modelo, df, factores_list, \'Y\', "tag")\n                mostrar_graficos_residuos_directo(modelo, df, \'F1\', \'Y\', "tag", "Residuos")\n\nwith tab11:\n    st.subheader("📌 Optimización Múltiple")\n    df_temp = obtener_datos_generales("omr", 4)\n    if df_temp is not None and len(df_temp.columns) >= 4:\n        col_f1 = st.selectbox("Factor 1:", df_temp.columns.tolist(), key="f1_omr")\n        cr = [c for c in df_temp.columns if c != col_f1]\n        col_f2 = st.selectbox("Factor 2:", cr, key="f2_omr")\n        cr = [c for c in cr if c != col_f2]\n        col_y1 = st.selectbox("Respuesta 1 (max):", cr, key="y1_omr")\n        cr = [c for c in cr if c != col_y1]\n        col_y2 = st.selectbox("Respuesta 2 (min):", cr, key="y2_omr")\n        if st.button("🔍 Optimizar", key="omr_btn"):\n            try:\n                df = df_temp.copy()\n                df.columns = [\'F1\' if c == col_f1 else \'F2\' if c == col_f2 else \'Y1\' if c == col_y1 else \'Y2\' if c == col_y2 else c for c in df.columns]\n                df = df[[\'F1\', \'F2\', \'Y1\', \'Y2\']]\n                m1 = ols(\'Y1 ~ F1 + F2 + I(F1**2) + I(F2**2) + F1:F2\', data=df).fit()\n                m2 = ols(\'Y2 ~ F1 + F2 + I(F1**2) + I(F2**2) + F1:F2\', data=df).fit()\n                grid = pd.DataFrame([(f1, f2) for f1 in np.linspace(-1.5, 1.5, 20) for f2 in np.linspace(-1.5, 1.5, 20)], columns=[\'F1\', \'F2\'])\n                grid[\'Y1\'] = m1.predict(grid)\n                grid[\'Y2\'] = m2.predict(grid)\n                grid[\'D\'] = (((grid[\'Y1\'] - grid[\'Y1\'].min()) / (grid[\'Y1\'].max() - grid[\'Y1\'].min())) * (1 - (grid[\'Y2\'] - grid[\'Y2\'].min()) / (grid[\'Y2\'].max() - grid[\'Y2\'].min()))) ** 0.5\n                opt = grid.loc[grid[\'D\'].idxmax()]\n                st.success(f"Óptimo: F1={opt[\'F1\']:.3f}, F2={opt[\'F2\']:.3f}, D={opt[\'D\']:.4f}")\n                st.markdown("---")\n                st.markdown("**Summary Statistics de la Respuesta 1 (Y1):**")\n                mostrar_summary_statistics(m1, df, [\'F1\', \'F2\'], \'Y1\', "omr_y1", incluir_descriptivas=False)\n            except Exception as e: st.error(f"❌ {e}")\n\nwith tab12:\n    st.subheader("📌 Diseños Añadidos")\n    df_temp = obtener_datos_generales("ad", 4)\n    if df_temp is not None and len(df_temp.columns) >= 4:\n        tipo_anova = selector_tipo_anova("ad")\n        c1 = st.selectbox("Comp 1:", df_temp.columns.tolist(), key="c1_ad")\n        cr = [c for c in df_temp.columns if c != c1]\n        c2 = st.selectbox("Comp 2:", cr, key="c2_ad")\n        cr = [c for c in cr if c != c2]\n        c3 = st.selectbox("Comp 3:", cr, key="c3_ad")\n        cr = [c for c in cr if c != c3]\n        col_r = st.selectbox("Respuesta:", cr, key="r_ad")\n        if st.button("🔍 Calcular", key="ad_btn"):\n            df = df_temp.copy()\n            df.columns = [\'C1\' if c == c1 else \'C2\' if c == c2 else \'C3\' if c == c3 else \'Y\' if c == col_r else c for c in df.columns]\n            df = df[[\'C1\', \'C2\', \'C3\', \'Y\']]\n            modelo, anova = analizar_anova_mejorada(df, \'Y ~ C1 + C2 + C3\', "Aditivos", "ad", "ad", tipo_anova)\n            if modelo is not None:\n                mostrar_summary_statistics(modelo, df, [\'C1\', \'C2\', \'C3\'], \'Y\', "ad")\n                mostrar_graficos_residuos_directo(modelo, df, \'C1\', \'Y\', "ad", "Residuos")\n\nwith tab13:\n    st.subheader("📌 Split-Plot")\n    df_temp = obtener_datos_generales("sp", 4)\n    if df_temp is not None and len(df_temp.columns) >= 4:\n        tipo_anova = selector_tipo_anova("sp")\n        f1 = st.selectbox("Factor 1:", df_temp.columns.tolist(), key="f1_sp")\n        cr = [c for c in df_temp.columns if c != f1]\n        f2 = st.selectbox("Factor 2:", cr, key="f2_sp")\n        cr = [c for c in cr if c != f2]\n        f3 = st.selectbox("Factor 3:", cr, key="f3_sp")\n        cr = [c for c in cr if c != f3]\n        col_r = st.selectbox("Respuesta:", cr, key="r_sp")\n        if st.button("🔍 Calcular", key="sp_btn"):\n            df = df_temp.copy()\n            df.columns = [\'F1\' if c == f1 else \'F2\' if c == f2 else \'F3\' if c == f3 else \'Y\' if c == col_r else c for c in df.columns]\n            df = df[[\'F1\', \'F2\', \'F3\', \'Y\']]\n            for c in [\'F1\', \'F2\', \'F3\']: df[c] = df[c].astype(\'category\')\n            modelo, anova = analizar_anova_mejorada(df, \'Y ~ F1 * F2 * F3\', "Split-Plot", "sp", "sp", tipo_anova)\n            if modelo is not None:\n                mostrar_summary_statistics(modelo, df, [\'F1\', \'F2\', \'F3\'], \'Y\', "sp")\n                fig_sc = graficar_sumas_cuadrados(anova)\n                if fig_sc: st.plotly_chart(fig_sc, use_container_width=True)\n                mostrar_graficos_residuos_directo(modelo, df, \'F1\', \'Y\', "sp", "Residuos")\n'
Path("app.py").write_text(APP_SOURCE, encoding="utf-8")
print("1/4 Aplicación app.py preparada.")

# Dependencias
req = Path("requirements.txt")
if req.exists():
    print("2/4 Verificando e instalando dependencias...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "streamlit", "pandas", "numpy", "scipy", "plotly",
                    "statsmodels", "requests"], check=True)

# Puerto libre
def puerto_libre(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", p)) != 0

PORT = 8501
while not puerto_libre(PORT):
    PORT += 1

# Configuración compatible con el proxy/iframe de Colab
cmd = [
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.port", str(PORT),
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.enableWebsocketCompression", "false",
    "--browser.gatherUsageStats", "false",
]
log_path = Path("streamlit_colab.log")
log_file = open(log_path, "w", encoding="utf-8")
proceso_streamlit = subprocess.Popen(
    cmd, stdout=log_file, stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"3/4 Iniciando Streamlit en el puerto {PORT}...")

# Comprobar el servidor y la página principal, no solo el proceso
listo = False
ultimo_error = None
for intento in range(90):
    if proceso_streamlit.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3) as r:
            if r.status == 200:
                listo = True
                break
    except Exception as e:
        ultimo_error = str(e)
    time.sleep(1)

if not listo:
    log_file.flush()
    print("No fue posible abrir la aplicación.")
    print("Último error:", ultimo_error)
    print("Registro de Streamlit:")
    print(log_path.read_text(encoding="utf-8", errors="replace")[-10000:])
    raise RuntimeError("Streamlit no quedó disponible. Revisa el registro anterior.")

log_file.flush()
print("4/4 Servidor HTTP activo. Mostrando calculadora...")
print("Si el visor integrado no carga, usa el enlace que aparece debajo.")
# Vista integrada de Colab
output.serve_kernel_port_as_iframe(PORT, height=1200)
# Enlace alternativo que abre el mismo servidor en una pestaña
output.serve_kernel_port_as_window(PORT)
